# FNA to GTF/GFF relationship mapping

## Initial relationship check

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Microbial reference sequence and annotation file matching and renaming script
Record any sequence overlap and save detailed match information for review
"""

import os
import gzip
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from multiprocessing import Pool, Manager
from tqdm import tqdm
import shutil

# Path configuration
BASE_DIR = Path("/path/to/project/data")
REFERENCE_DIR = BASE_DIR / "reference"
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
GTF_DIR = ANNOTATION_DIR / "gtf"
GFF_DIR = ANNOTATION_DIR / "gff"
GTF_OUTPUT_DIR = ANNOTATION_DIR / "gtf_mic"
GFF_OUTPUT_DIR = ANNOTATION_DIR / "gff_mic"
TAXID_HIERARCHY = ANNOTATION_DIR / "V7_taxid_hierarchy.txt"
ASSEMBLY_INFO = ANNOTATION_DIR / "V7_matched_assembly_info.xlsx"
OUTPUT_MAPPING = ANNOTATION_DIR / "fna_annotation_mapping.tsv"

# Number of parallel cores
NUM_CORES = 64


def sanitize_species_name(species_name):
    """Convert species_scientific_name to filename format"""
    sanitized = re.sub(r'[\s/\\:*-"<>|]+', '_', species_name)
    return sanitized


def extract_seq_ids_from_fna(fna_path):
    """Extract all reference sequence IDs from the FNA file"""
    seq_ids = set()
    try:
        if str(fna_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(fna_path, mode) as f:
                for line in f:
                    if line.startswith('>'):
                        seq_id = line[1:].split()[0]
                        seq_ids.add(seq_id)
    except Exception as e:
        print(f"Error reading {fna_path}: {e}")
        return seq_ids


def extract_seq_ids_from_annotation(annotation_path):
    """Extract all reference sequence IDs from the GTF/GFF file (first column)"""
    seq_ids = set()
    try:
        if str(annotation_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(annotation_path, mode) as f:
                for line in f:
                    if line.startswith('#'):
                        continue
                    parts = line.strip().split('\t')
                    if len(parts) > 0:
                        seq_ids.add(parts[0])
    except Exception as e:
        print(f"Error reading {annotation_path}: {e}")
        return seq_ids


def calculate_match_details(fna_ids, ann_ids):
    """Calculate detailed match information"""
    if not fna_ids or not ann_ids:
        return None
    intersection = fna_ids & ann_ids
    fna_only = fna_ids - ann_ids
    ann_only = ann_ids - fna_ids
    # Calculate different overlap-rate metrics
    overlap_union = len(intersection) / len(fna_ids | ann_ids) if (fna_ids | ann_ids) else 0
    overlap_fna = len(intersection) / len(fna_ids) if fna_ids else 0
    overlap_ann = len(intersection) / len(ann_ids) if ann_ids else 0
    return {
        'intersection_count': len(intersection),
        'fna_total_seqs': len(fna_ids),
        'ann_total_seqs': len(ann_ids),
        'fna_only_count': len(fna_only),
        'ann_only_count': len(ann_only),
        'overlap_union_ratio': overlap_union, # intersection/union
        'overlap_fna_ratio': overlap_fna, # intersection/FNA total
        'overlap_ann_ratio': overlap_ann, # intersection/annotation total
        'intersection_ids': ','.join(sorted(intersection)),
        'fna_only_ids': ','.join(sorted(fna_only)),
        'ann_only_ids': ','.join(sorted(ann_only))
    }


def extract_gcf_from_filename(filename):
    """Extract the GCF accession from the filename"""
    match = re.search(r'(GCF_\d+\.\d+)', filename)
    return match.group(1) if match else None


def process_fna_file(args):
    """Process one FNA file and find the corresponding annotation file"""
    fna_path, species_to_taxids, gcf_to_taxid, gtf_files_by_taxid, gff_files_by_taxid = args
    results = []
    fna_name = fna_path.stem.replace('.fna', '')
    # Extract sequence IDs from the FNA file
    fna_seq_ids = extract_seq_ids_from_fna(fna_path)
    if not fna_seq_ids:
        return results
    # taxid list
    taxids = species_to_taxids.get(fna_name, [])
    if not taxids:
        print(f"Warning: No taxid found for {fna_name}")
        return results
    # all GTFandGFFfile
    candidate_gtf_files = []
    candidate_gff_files = []
    for taxid in taxids:
        candidate_gtf_files.extend(gtf_files_by_taxid.get(taxid, []))
        candidate_gff_files.extend(gff_files_by_taxid.get(taxid, []))
        # processGTFfile - allwith
        gtf_matches = []
        for gtf_path in candidate_gtf_files:
            gtf_seq_ids = extract_seq_ids_from_annotation(gtf_path)
            match_details = calculate_match_details(fna_seq_ids, gtf_seq_ids)
            if match_details and match_details['intersection_count'] > 0: # onlywith coverage record
                gtf_matches.append({
                    'path': gtf_path,
                    'details': match_details
                })
                # count, matched
                if gtf_matches:
                    gtf_matches.sort(key=lambda x: x['details']['intersection_count'], reverse=True)
                    best_gtf = gtf_matches[0]
                    result = {
                        'fna_file': fna_path.name,
                        'species_name': fna_name,
                        'annotation_type': 'gtf',
                        'original_annotation_file': best_gtf['path'].name,
                        'gcf_accession': extract_gcf_from_filename(best_gtf['path'].name),
                        'matched_taxids': ','.join(map(str, taxids)),
                        **best_gtf['details']
                    }
                    results.append(result)
                    # processGFFfile - allwith
                    gff_matches = []
                    for gff_path in candidate_gff_files:
                        gff_seq_ids = extract_seq_ids_from_annotation(gff_path)
                        match_details = calculate_match_details(fna_seq_ids, gff_seq_ids)
                        if match_details and match_details['intersection_count'] > 0: # onlywith coverage record
                            gff_matches.append({
                                'path': gff_path,
                                'details': match_details
                            })
                            # count, matched
                            if gff_matches:
                                gff_matches.sort(key=lambda x: x['details']['intersection_count'], reverse=True)
                                best_gff = gff_matches[0]
                                result = {
                                    'fna_file': fna_path.name,
                                    'species_name': fna_name,
                                    'annotation_type': 'gff',
                                    'original_annotation_file': best_gff['path'].name,
                                    'gcf_accession': extract_gcf_from_filename(best_gff['path'].name),
                                    'matched_taxids': ','.join(map(str, taxids)),
                                    **best_gff['details']
                                }
                                results.append(result)
                                return results


def main():
    print("=" * 80)
    print("microbe sequence annotation file matched ")
    print("Summary: only with sequence record, detailed record matched information")
    print("=" * 80)
    # Create output directory
    GTF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    GFF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    # 1. read taxid information
    print("\n[1/6] read taxid information...")
    taxid_df = pd.read_csv(TAXID_HIERARCHY, sep='\t')
    # species_scientific_nametotax_idmapping( )
    species_to_taxids = defaultdict(list)
    for _, row in taxid_df.iterrows():
        species_name = sanitize_species_name(row['species_scientific_name'])
        tax_id = int(row['tax_id'])
        species_to_taxids[species_name].append(tax_id)
        print(f" load {len(species_to_taxids)} species taxid information")
        # 2. readassemblyinformation
        print("\n[2/6] readassemblyinformation...")
        assembly_df = pd.read_excel(ASSEMBLY_INFO)
        # GCFtotaxidmapping
        gcf_to_taxid = {}
        for _, row in assembly_df.iterrows():
            gcf = row['assembly_accession']
            taxid = int(row['taxid'])
            gcf_to_taxid[gcf] = taxid
            print(f" load {len(gcf_to_taxid)} GCFtaxidinformation")
            # 3. annotation file
            print("\n[3/6] annotation file...")
            # taxidtoGTFfilemapping
            gtf_files_by_taxid = defaultdict(list)
            gtf_files = list(GTF_DIR.glob("*.gtf*"))
            for gtf_file in tqdm(gtf_files, desc=" processGTFfile"):
                gcf = extract_gcf_from_filename(gtf_file.name)
                if gcf and gcf in gcf_to_taxid:
                    taxid = gcf_to_taxid[gcf]
                    gtf_files_by_taxid[taxid].append(gtf_file)
                    print(f" found {len(gtf_files)} GTFfile, coverage {len(gtf_files_by_taxid)} taxid")
                    # taxidtoGFFfilemapping
                    gff_files_by_taxid = defaultdict(list)
                    gff_files = list(GFF_DIR.glob("*.gff*"))
                    for gff_file in tqdm(gff_files, desc=" processGFFfile"):
                        gcf = extract_gcf_from_filename(gff_file.name)
                        if gcf and gcf in gcf_to_taxid:
                            taxid = gcf_to_taxid[gcf]
                            gff_files_by_taxid[taxid].append(gff_file)
                            print(f" found {len(gff_files)} GFFfile, coverage {len(gff_files_by_taxid)} taxid")
                            # 4. fnafile(only .fnaand.fna.gz, BWA file)
                            print("\n[4/6] fnafile...")
                            fna_files = []
                            # only .fnaand.fna.gzfile
                            fna_files.extend(REFERENCE_DIR.glob("*.fna"))
                            fna_files.extend(REFERENCE_DIR.glob("*.fna.gz"))
                            # with coverage
                            fna_files = list(set(fna_files))
                            print(f" found {len(fna_files)} FNAfile( BWA file)")
                            # 5. rows process matched
                            print(f"\n[5/6] {NUM_CORES} rows matched...")
                            # Processing section
                            process_args = [
                                (fna_file, species_to_taxids, gcf_to_taxid, gtf_files_by_taxid, gff_files_by_taxid)
                                for fna_file in fna_files
                            ]
                            # rows process
                            all_results = []
                            with Pool(NUM_CORES) as pool:
                                for results in tqdm(pool.imap_unordered(process_fna_file, process_args), total=len(process_args), desc=" matched "):
                                all_results.extend(results)
                                print(f" completed matched, found {len(all_results)} matched ")
                                # 6. save results file
                                print("\n[6/6] save results file...")
                                if not all_results:
                                    print(" warning: with found matched!")
                                    return
                                # forDataFrame
                                results_df = pd.DataFrame(all_results)
                                # column column, information first
                                column_order = [
                                    'fna_file', 'species_name', 'annotation_type', 'original_annotation_file', 'gcf_accession',
                                    'intersection_count', 'fna_total_seqs', 'ann_total_seqs',
                                    'fna_only_count', 'ann_only_count',
                                    'overlap_union_ratio', 'overlap_fna_ratio', 'overlap_ann_ratio',
                                    'matched_taxids', 'intersection_ids', 'fna_only_ids', 'ann_only_ids'
                                ]
                                results_df = results_df[column_order]
                                # copy files
                                renamed_count = 0
                                for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc=" file"):
                                    species_name = row['species_name']
                                    annotation_type = row['annotation_type']
                                    original_file = row['original_annotation_file']
                                    if annotation_type == 'gtf':
                                        source_path = GTF_DIR / original_file
                                        new_filename = f"{species_name}_{original_file}"
                                        dest_path = GTF_OUTPUT_DIR / new_filename
                                    else: # gff
                                        source_path = GFF_DIR / original_file
                                        new_filename = f"{species_name}_{original_file}"
                                        dest_path = GFF_OUTPUT_DIR / new_filename
                                        if source_path.exists():
                                            shutil.copy2(source_path, dest_path)
                                            renamed_count += 1
                                            # save mapping
                                            results_df.to_csv(OUTPUT_MAPPING, sep='\t', index=False)
                                            print(f"\n{'=' * 80}")
                                            print("Processing completed!")
                                            print(f"{'=' * 80}")
                                            print(f" FNAfileSummary: {len(fna_files)}")
                                            print(f"successmatchedSummary: {len(all_results)}")
                                            print(f" fileSummary: {renamed_count}")
                                            print(f"\noutputSummary:")
                                            print(f" - GTFfile: {GTF_OUTPUT_DIR}")
                                            print(f" - GFFfile: {GFF_OUTPUT_DIR}")
                                            print(f" - mappingSummary: {OUTPUT_MAPPING}")
                                            # statistics
                                            print(f"\nmatchedstatistics:")
                                            print(f" - GTFmatched: {len(results_df[results_df['annotation_type'] == 'gtf'])} ")
                                            print(f" - GFFmatched: {len(results_df[results_df['annotation_type'] == 'gff'])} ")
                                            # statistics
                                            print(f"\n statistics:")
                                            print(f" - average count: {results_df['intersection_count'].mean():.1f} sequence")
                                            print(f" - averageFNA sequence: {results_df['fna_total_seqs'].mean():.1f} ")
                                            print(f" - average annotation file sequence: {results_df['ann_total_seqs'].mean():.1f} ")
                                            print(f" - average (intersection/union): {results_df['overlap_union_ratio'].mean():.2%}")
                                            print(f" - averageFNAcoverageSummary: {results_df['overlap_fna_ratio'].mean():.2%}")
                                            print(f" - average annotation coverageSummary: {results_df['overlap_ann_ratio'].mean():.2%}")
                                            # matched and matched statistics
                                            perfect_match = results_df[results_df['overlap_union_ratio'] >= 0.99]
                                            good_match = results_df[(results_df['overlap_union_ratio'] >= 0.8) & (results_df['overlap_union_ratio'] < 0.99)]
                                            partial_match = results_df[results_df['overlap_union_ratio'] < 0.8]
                                            print(f"\nmatchedSummary:")
                                            print(f" - matched (>=99%): {len(perfect_match)} ")
                                            print(f" - matched (80-99%): {len(good_match)} ")
                                            print(f" - matched (<80%): {len(partial_match)} ")
                                            print(f"\nNote: Summary: {OUTPUT_MAPPING} detailed matched information")
                                            print(f" - intersection_count: sequence ")
                                            print(f" - fna_only_count: only FNAinsequence ")
                                            print(f" - ann_only_count: only annotationfileinsequence ")
                                            print(f" - overlap_fna_ratio: FNAinwith annotation coverage")


if __name__ == "__main__":
    main()

## Check microbes with FNA but no annotation

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Find microbes with FNA files but no annotation files
"""

import pandas as pd
from pathlib import Path

# Path configuration
BASE_DIR = Path("/path/to/project/data")
REFERENCE_DIR = BASE_DIR / "reference"
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
MAPPING_FILE = ANNOTATION_DIR / "report/fna_annotation_mapping.tsv"
OUTPUT_FILE = ANNOTATION_DIR / "report/missing_annotations_report.tsv"

def main():
    print("=" * 80)
    print("findmissing annotationmicrobe")
    print("=" * 80)
    # 1. readallFNAfile
    print("\n[1/3] Scan all FNA files...")
    fna_files = []
    fna_files.extend(REFERENCE_DIR.glob("*.fna"))
    fna_files.extend(REFERENCE_DIR.glob("*.fna.gz"))
    fna_files = list(set(fna_files))
    # extractspecies name(filenameremove extension)
    all_species = set()
    for fna_file in fna_files:
        species_name = fna_file.stem.replace('.fna', '')
        all_species.add(species_name)
        print(f" found {len(all_species)} speciesFNAfile")
        # 2. readmatchedresults
        print("\n[2/3] readmatchedresults...")
        mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
        # extractwithannotationspecies(deduplicate)
        species_with_gtf = set(mapping_df[mapping_df['annotation_type'] == 'gtf']['species_name'].unique())
        species_with_gff = set(mapping_df[mapping_df['annotation_type'] == 'gff']['species_name'].unique())
        species_with_any_annotation = species_with_gtf | species_with_gff
        print(f" species with GTF annotation: {len(species_with_gtf)} ")
        print(f" species with GFF annotation: {len(species_with_gff)} ")
        print(f" species with any annotation: {len(species_with_any_annotation)} ")
        # 3. missing annotation species
        print("\n[3/3] Analyze missingness...")
        missing_gtf = all_species - species_with_gtf
        missing_gff = all_species - species_with_gff
        missing_all = all_species - species_with_any_annotation
        print(f"\n statistics:")
        print(f" - GTFspecies: {len(missing_gtf)} ")
        print(f" - GFFspecies: {len(missing_gff)} ")
        print(f" - with annotation species: {len(missing_all)} ")
        # createdetailedreport
        report_data = []
        for species in sorted(all_species):
            has_gtf = species in species_with_gtf
            has_gff = species in species_with_gff
            # with annotation, detailed information
            gtf_info = ""
            gff_info = ""
            if has_gtf:
                gtf_rows = mapping_df[(mapping_df['species_name'] == species) & (mapping_df['annotation_type'] == 'gtf')]
                if len(gtf_rows) > 0:
                    row = gtf_rows.iloc[0]
                    gtf_info = f"{row['gcf_accession']} (overlap: {row['overlap_fna_ratio']:.2%})"
                    if has_gff:
                        gff_rows = mapping_df[(mapping_df['species_name'] == species) & (mapping_df['annotation_type'] == 'gff')]
                        if len(gff_rows) > 0:
                            row = gff_rows.iloc[0]
                            gff_info = f"{row['gcf_accession']} (overlap: {row['overlap_fna_ratio']:.2%})"
                            report_data.append({
                                'species_name': species,
                                'has_gtf': 'Yes' if has_gtf else 'No',
                                'has_gff': 'Yes' if has_gff else 'No',
                                'has_any_annotation': 'Yes' if (has_gtf or has_gff) else 'No',
                                'gtf_info': gtf_info,
                                'gff_info': gff_info
                            })
                            # savereport
                            report_df = pd.DataFrame(report_data)
                            report_df.to_csv(OUTPUT_FILE, sep='\t', index=False)
                            # with annotation species
                            if missing_all:
                                print(f"\n{'=' * 80}")
                                print(f" with annotation {len(missing_all)} species:")
                                print(f"{'=' * 80}")
                                for species in sorted(missing_all):
                                    print(f" - {species}")
                                else:
                                    print("\nOK !all species with coverageAnnotation format")
                                    # onlywithGTF withGFF
                                    only_gtf = species_with_gtf - species_with_gff
                                    if only_gtf:
                                        print(f"\nonlywithGTF withGFF {len(only_gtf)} species:")
                                        for species in sorted(only_gtf):
                                            print(f" - {species}")
                                            # onlywithGFF withGTF
                                            only_gff = species_with_gff - species_with_gtf
                                            if only_gff:
                                                print(f"\nonlywithGFF withGTF {len(only_gff)} species:")
                                                for species in sorted(only_gff):
                                                    print(f" - {species}")
                                                    print(f"\n{'=' * 80}")
                                                    print(f"detailedreportsavedSummary: {OUTPUT_FILE}")
                                                    print(f"{'=' * 80}")
                                                    # statistics
                                                    print(f"\nSummary:")
                                                    print(f" speciesSummary: {len(all_species)}")
                                                    print(f" withannotation: {len(species_with_any_annotation)} ({len(species_with_any_annotation)/len(all_species)*100:.1f}%)")
                                                    print(f" noannotation: {len(missing_all)} ({len(missing_all)/len(all_species)*100:.1f}%)")
                                                    print(f" withGTFandGFF: {len(species_with_gtf & species_with_gff)} ({len(species_with_gtf & species_with_gff)/len(all_species)*100:.1f}%)")


if __name__ == "__main__":
    main()

# FNA to annotation protein sequence ID mapping

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Extract protein sequence IDs and gene names from GTF/GFF annotation files
"""

import os
import gzip
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from multiprocessing import Pool
from tqdm import tqdm

# Path configuration
BASE_DIR = Path("/path/to/project/data")
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
GTF_OUTPUT_DIR = ANNOTATION_DIR / "gtf_mic"
GFF_OUTPUT_DIR = ANNOTATION_DIR / "gff_mic"
MAPPING_FILE = ANNOTATION_DIR / "report/fna_annotation_mapping.tsv"
TAXID_HIERARCHY = ANNOTATION_DIR / "V7_taxid_hierarchy.txt"

# Output files
OUTPUT_PROTEIN_GENE = ANNOTATION_DIR / "report/fna_protein_gene_mapping.tsv"
OUTPUT_SUMMARY = ANNOTATION_DIR / "report/protein_gene_summary.tsv"

# Number of parallel cores
NUM_CORES = 64


def parse_gtf_attributes(attr_string):
    """
    Parse the GTF-format attribute string
    GTFformat: key "value"; key2 "value2";
    """
    attributes = {}
    # matched key "value"
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes


def parse_gff_attributes(attr_string):
    """
    Parse the GFF-format attribute string
    GFFformat: key=value;key2=value2
    """
    attributes = {}
    if not attr_string or attr_string.strip() == '.':
        return attributes
    # Processing section
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            # URL
            value = value.replace('%2C', ',').replace('%3B', ';')
            attributes[key.strip()] = value.strip()
            return attributes


def extract_from_gtf(gtf_path, fna_seq_ids):
    """
    fromGTFfileinextractproteinsequenceIDandgene
    return: [(seq_id, protein_id, gene_name, gene_id, feature_type), ...]
    """
    results = []
    seen_proteins = set() #
    try:
        if str(gtf_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(gtf_path, mode) as f:
                for line in f:
                    # skipannotationrows
                    if line.startswith('#'):
                        continue
                    parts = line.strip().split('\t')
                    if len(parts) < 9:
                        continue
                    seq_id = parts[0]
                    feature_type = parts[2]
                    attributes_str = parts[8]
                    # onlyprocess FNAinexistssequenceID
                    if seq_id not in fna_seq_ids:
                        continue
                    # Processing section
                    attrs = parse_gtf_attributes(attributes_str)
                    # extractprotein_id( CDS in)
                    protein_id = attrs.get('protein_id', '')
                    gene_name = attrs.get('gene_name', attrs.get('gene', ''))
                    gene_id = attrs.get('gene_id', '')
                    transcript_id = attrs.get('transcript_id', '')
                    # withprotein_id, record
                    if protein_id:
                        key = (seq_id, protein_id, gene_name, gene_id)
                        if key not in seen_proteins:
                            seen_proteins.add(key)
                            results.append({
                                'seq_id': seq_id,
                                'protein_id': protein_id,
                                'gene_name': gene_name,
                                'gene_id': gene_id,
                                'transcript_id': transcript_id,
                                'feature_type': feature_type
                            })
                            # withprotein_id, gene record gene information
                        elif feature_type == 'gene' and gene_id:
                            key = (seq_id, '', gene_name, gene_id)
                            if key not in seen_proteins:
                                seen_proteins.add(key)
                                results.append({
                                    'seq_id': seq_id,
                                    'protein_id': '',
                                    'gene_name': gene_name,
                                    'gene_id': gene_id,
                                    'transcript_id': '',
                                    'feature_type': feature_type
                                })
    except Exception as e:
        print(f"Error reading GTF {gtf_path}: {e}")
        return results


def extract_from_gff(gff_path, fna_seq_ids):
    """
    fromGFFfileinextractproteinsequenceIDandgene
    return: [(seq_id, protein_id, gene_name, gene_id, feature_type), ...]
    """
    results = []
    seen_proteins = set()
    try:
        if str(gff_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(gff_path, mode) as f:
                for line in f:
                    # skipannotationrowsandFASTA
                    if line.startswith('#') or line.startswith('>'):
                        continue
                    parts = line.strip().split('\t')
                    if len(parts) < 9:
                        continue
                    seq_id = parts[0]
                    feature_type = parts[2]
                    attributes_str = parts[8]
                    # onlyprocess FNAinexistssequenceID
                    if seq_id not in fna_seq_ids:
                        continue
                    # Processing section
                    attrs = parse_gff_attributes(attributes_str)
                    # GFFinprotein_id in
                    protein_id = (attrs.get('protein_id', '') or attrs.get('Dbxref', '').split(':')[-1] if 'Dbxref' in attrs else '')
                    # geneName, gene, gene_name in
                    gene_name = (attrs.get('gene', '') or attrs.get('Name', '') or attrs.get('gene_name', ''))
                    gene_id = attrs.get('ID', attrs.get('gene_id', ''))
                    parent = attrs.get('Parent', '')
                    # recordwithprotein_idorgeneinformation
                    if protein_id or (feature_type == 'gene' and gene_id):
                        key = (seq_id, protein_id, gene_name, gene_id)
                        if key not in seen_proteins:
                            seen_proteins.add(key)
                            results.append({
                                'seq_id': seq_id,
                                'protein_id': protein_id,
                                'gene_name': gene_name,
                                'gene_id': gene_id,
                                'parent': parent,
                                'feature_type': feature_type
                            })
    except Exception as e:
        print(f"Error reading GFF {gff_path}: {e}")
        return results


def process_single_mapping(args):
    """process FNA-annotation file mapping"""
    row_dict, gtf_output_dir, gff_output_dir, taxid_to_name = args
    fna_file = row_dict['fna_file']
    species_name = row_dict['species_name']
    annotation_type = row_dict['annotation_type']
    original_annotation_file = row_dict['original_annotation_file']
    matched_taxids = row_dict.get('matched_taxids', '')
    # build afterfilepath
    renamed_file = f"{species_name}_{original_annotation_file}"
    if annotation_type == 'gtf':
        ann_path = gtf_output_dir / renamed_file
    else:
        ann_path = gff_output_dir / renamed_file
        if not ann_path.exists():
            print(f"Warning: Annotation file not found: {ann_path}")
            return []
        # FNAsequenceID(fromintersection_ids )
        fna_seq_ids = set()
        if 'intersection_ids' in row_dict and row_dict['intersection_ids']:
            fna_seq_ids = set(row_dict['intersection_ids'].split(','))
            if not fna_seq_ids:
                print(f"Warning: No sequence IDs found for {fna_file}")
                return []
            # extractinformation
            if annotation_type == 'gtf':
                extracted_data = extract_from_gtf(ann_path, fna_seq_ids)
            else:
                extracted_data = extract_from_gff(ann_path, fna_seq_ids)
                # taxid microbe
                taxid_list = [tid.strip() for tid in matched_taxids.split(',')] if matched_taxids else []
                microbe_names = []
                for tid in taxid_list:
                    if tid in taxid_to_name:
                        microbe_names.append(taxid_to_name[tid])
                        microbe_name_str = '; '.join(microbe_names) if microbe_names else ''
                        # information
                        for item in extracted_data:
                            item['fna_file'] = fna_file
                            item['species_name'] = species_name
                            item['annotation_type'] = annotation_type
                            item['annotation_file'] = renamed_file
                            item['taxids'] = matched_taxids
                            item['microbe_scientific_name'] = microbe_name_str
                            return extracted_data


def main():
    print("=" * 80)
    print("fromGTF/GFFfileinextractproteinsequenceIDandgene ")
    print("=" * 80)
    # 1. read taxid information
    print("\n[1/5] read taxid information...")
    if not TAXID_HIERARCHY.exists():
        print(f"error: taxid file does not exist: {TAXID_HIERARCHY}")
        return
    taxid_df = pd.read_csv(TAXID_HIERARCHY, sep='\t')
    # taxidtospecies mapping
    taxid_to_name = {}
    for _, row in taxid_df.iterrows():
        tax_id = str(row['tax_id'])
        species_name = row['species_scientific_name']
        taxid_to_name[tax_id] = species_name
        print(f" load {len(taxid_to_name)} taxid species information")
        # 2. readmappingfile
        print("\n[2/5] readFNA-annotationmappingfile...")
        if not MAPPING_FILE.exists():
            print(f"error: mapping file does not exist: {MAPPING_FILE}")
            print(" rows FNA to GTF/GFF relationship mapping.py generatemappingfile")
            return
        mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
        print(f" load {len(mapping_df)} mapping record")
        # 3. rows process
        print(f"\n[3/5] process {len(mapping_df)} annotation file...")
        process_args = [
            (row.to_dict(), GTF_OUTPUT_DIR, GFF_OUTPUT_DIR, taxid_to_name)
            for _, row in mapping_df.iterrows()
        ]
        # 4. rowsextractinformation
        print(f"\n[4/5] {NUM_CORES} rowsextractproteinandgeneinformation...")
        all_results = []
        with Pool(NUM_CORES) as pool:
            for results in tqdm(pool.imap_unordered(process_single_mapping, process_args),
                total=len(process_args), desc=" extract "):
            all_results.extend(results)
            print(f" extractto {len(all_results)} protein/gene record")
            if not all_results:
                print("\nwarning: withextractto data!")
                print("Summary: ")
                print("1. annotationfilein withprotein_idinformation")
                print("2. sequenceID matched")
                print("3. GTF/GFFfileformat ")
                return
            # 5. saveresults
            print("\n[5/5] saveresults...")
            # forDataFrame
            results_df = pd.DataFrame(all_results)
            # column, information first
            column_order = [
                'fna_file', 'species_name', 'taxids', 'microbe_scientific_name',
                'seq_id', 'protein_id', 'gene_name', 'gene_id',
                'annotation_type', 'annotation_file', 'feature_type'
            ]
            # column(with coverage)
            other_columns = [col for col in results_df.columns if col not in column_order]
            column_order.extend(other_columns)
            # onlydoes not existcolumn
            column_order = [col for col in column_order if col in results_df.columns]
            results_df = results_df[column_order]
            # savedetailedinformation
            results_df.to_csv(OUTPUT_PROTEIN_GENE, sep='\t', index=False)
            print(f" detailed results savedto: {OUTPUT_PROTEIN_GENE}")
            # generatestatistics
            print("\ngeneratestatistics...")
            summary_data = []
            for fna_file in results_df['fna_file'].unique():
                fna_data = results_df[results_df['fna_file'] == fna_file]
                # statisticsGTFandGFF
                for ann_type in fna_data['annotation_type'].unique():
                    type_data = fna_data[fna_data['annotation_type'] == ann_type]
                    summary_data.append({
                        'fna_file': fna_file,
                        'species_name': type_data['species_name'].iloc[0],
                        'taxids': type_data['taxids'].iloc[0],
                        'microbe_scientific_name': type_data['microbe_scientific_name'].iloc[0],
                        'annotation_type': ann_type,
                        'annotation_file': type_data['annotation_file'].iloc[0],
                        'total_proteins': len(type_data[type_data['protein_id'] != '']),
                        'total_genes': len(type_data['gene_id'].unique()),
                        'proteins_with_gene_name': len(type_data[(type_data['protein_id'] != '') & (type_data['gene_name'] != '')]),
                        'unique_seq_ids': len(type_data['seq_id'].unique()),
                        'unique_protein_ids': len(type_data[type_data['protein_id'] != '']['protein_id'].unique()),
                        'unique_gene_names': len(type_data[type_data['gene_name'] != '']['gene_name'].unique())
                    })
                    summary_df = pd.DataFrame(summary_data)
                    summary_df.to_csv(OUTPUT_SUMMARY, sep='\t', index=False)
                    print(f" statisticssavedto: {OUTPUT_SUMMARY}")
                    # statistics
                    print(f"\n{'=' * 80}")
                    print("extractcompleted!statistics: ")
                    print(f"{'=' * 80}")
                    print(f"total records: {len(results_df)}")
                    print(f" FNAfile: {results_df['fna_file'].nunique()} ")
                    print(f" species: {results_df['species_name'].nunique()} ")
                    print(f" taxid: {len(set(','.join(results_df['taxids'].unique()).split(',')))} ")
                    print(f"\n annotation statistics:")
                    for ann_type in results_df['annotation_type'].unique():
                        type_data = results_df[results_df['annotation_type'] == ann_type]
                        proteins_with_id = type_data[type_data['protein_id'] != '']
                        print(f" {ann_type.upper()}:")
                        print(f" - fileSummary: {type_data['annotation_file'].nunique()}")
                        print(f" - protein record: {len(proteins_with_id)}")
                        print(f" - uniqueproteinID: {proteins_with_id['protein_id'].nunique()}")
                        print(f" - unique geneSummary: {type_data[type_data['gene_name'] != '']['gene_name'].nunique()}")
                        print(f" - with gene protein: {len(proteins_with_id[proteins_with_id['gene_name'] != ''])}")
                        print(f"\nSummary:")
                        feature_counts = results_df['feature_type'].value_counts()
                        for feature, count in feature_counts.head(10).items():
                            print(f" {feature}: {count}")
                            print(f"\nOutput files:")
                            print(f" - detaileddata: {OUTPUT_PROTEIN_GENE}")
                            print(f" - statistics: {OUTPUT_SUMMARY}")
                            print(f"\nNote: Summary:")
                            print(f" - detailed data proteins information( taxid and microbe )")
                            print(f" - statistics FNAfileandannotation statistics")
                            print(f" - passed seq_id toFNAfilein sequence")
                            print(f" - taxids taxid( )")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Validate GTF and GFF annotation files
Check file paths, formats, naming rules, and related metadata
"""

import gzip
import re
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

# Path configuration
BASE_DIR = Path("/path/to/project/data")
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
MAPPING_FILE = ANNOTATION_DIR / "report/fna_annotation_mapping.tsv"
GTF_OUTPUT_DIR = ANNOTATION_DIR / "gtf_mic"
GFF_OUTPUT_DIR = ANNOTATION_DIR / "gff_mic"


def detect_file_format(file_path, sample_lines=20):
    """
    Detect the actual file format (GTF or GFF)
    Infer the format by analyzing the attributes column
    """
    gtf_indicators = 0 # "value" format rows
    gff_indicators = 0 # key=value format rows
    try:
        if str(file_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(file_path, mode) as f:
                for i, line in enumerate(f):
                    if i >= sample_lines:
                        break
                    if line.startswith('#'):
                        continue
                    parts = line.strip().split('\t')
                    if len(parts) >= 9:
                        attr_string = parts[8]
                        # checkGTFSummary:
                        if '"' in attr_string and re.search(r'\w+\s+"[^"]+"', attr_string):
                            gtf_indicators += 1
                            # checkGFFSummary: with coverageor
                            if '=' in attr_string and attr_string.count('=') > attr_string.count('"'):
                                gff_indicators += 1
                                if gtf_indicators > gff_indicators:
                                    return 'GTF', gtf_indicators, gff_indicators
                                elif gff_indicators > gtf_indicators:
                                    return 'GFF', gtf_indicators, gff_indicators
                                else:
                                    return 'UNKNOWN', gtf_indicators, gff_indicators
    except Exception as e:
        return 'ERROR', 0, 0


def check_file_extensions():
    """Check file extensions in the GTF and GFF directories"""
    print("\n" + "=" * 80)
    print("File extension check")
    print("=" * 80)
    # checkGTFdirectory
    print(f"\nGTFdirectory: {GTF_OUTPUT_DIR}")
    gtf_files = list(GTF_OUTPUT_DIR.glob("*"))
    gtf_extensions = defaultdict(int)
    for f in gtf_files:
        ext = ''.join(f.suffixes) # allafter, .gtf.gz
        gtf_extensions[ext] += 1
        print(f" fileSummary: {len(gtf_files)}")
        print(f" Summary:")
        for ext, count in sorted(gtf_extensions.items()):
            print(f" {ext if ext else '(no )'}: {count} ")
            # checkGFFdirectory
            print(f"\nGFFdirectory: {GFF_OUTPUT_DIR}")
            gff_files = list(GFF_OUTPUT_DIR.glob("*"))
            gff_extensions = defaultdict(int)
            for f in gff_files:
                ext = ''.join(f.suffixes)
                gff_extensions[ext] += 1
                print(f" fileSummary: {len(gff_files)}")
                print(f" Summary:")
                for ext, count in sorted(gff_extensions.items()):
                    print(f" {ext if ext else '(no )'}: {count} ")


def validate_mapping_files():
    """ mappingfileinallfile does not exist formatcorrect"""
    print("\n" + "=" * 80)
    print("mapping file ")
    print("=" * 80)
    # readmappingfile
    mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
    print(f"\nmappingrecordSummary: {len(mapping_df)}")
    # statistics
    gtf_mappings = mapping_df[mapping_df['annotation_type'] == 'gtf']
    gff_mappings = mapping_df[mapping_df['annotation_type'] == 'gff']
    print(f" GTFmapping: {len(gtf_mappings)} ")
    print(f" GFFmapping: {len(gff_mappings)} ")
    # eachmapping
    results = []
    print("\n file exists and format...")
    for _, row in tqdm(mapping_df.iterrows(), total=len(mapping_df)):
        fna_file = row['fna_file']
        species_name = row['species_name']
        annotation_type = row['annotation_type']
        annotation_file = row['original_annotation_file']
        # buildpath
        fna_path = BASE_DIR / "reference" / fna_file
        if annotation_type == 'gtf':
            renamed_file = f"{species_name}_{annotation_file}"
            annotation_path = GTF_OUTPUT_DIR / renamed_file
            expected_format = 'GTF'
        else:
            renamed_file = f"{species_name}_{annotation_file}"
            annotation_path = GFF_OUTPUT_DIR / renamed_file
            expected_format = 'GFF'
            # check file exists
            fna_exists = fna_path.exists()
            ann_exists = annotation_path.exists()
            # actualformat
            if ann_exists:
                detected_format, gtf_score, gff_score = detect_file_format(annotation_path)
            else:
                detected_format = 'FILE_NOT_FOUND'
                gtf_score = 0
                gff_score = 0
                # format matched
                format_match = (detected_format == expected_format)
                results.append({
                    'fna_file': fna_file,
                    'species_name': species_name,
                    'annotation_type': annotation_type,
                    'annotation_file': annotation_file,
                    'renamed_file': renamed_file,
                    'fna_exists': fna_exists,
                    'annotation_exists': ann_exists,
                    'expected_format': expected_format,
                    'detected_format': detected_format,
                    'format_match': format_match,
                    'gtf_score': gtf_score,
                    'gff_score': gff_score
                })
                # forDataFrame
                results_df = pd.DataFrame(results)
                # statisticsresults
                print("\n" + "=" * 80)
                print(" results")
                print("=" * 80)
                # file exists statistics
                fna_missing = (~results_df['fna_exists']).sum()
                ann_missing = (~results_df['annotation_exists']).sum()
                print(f"\nfile existsSummary:")
                print(f" FNAfileSummary: {fna_missing} ")
                print(f" annotation fileSummary: {ann_missing} ")
                if fna_missing > 0:
                    print(f"\n FNAfile:")
                    for fna in results_df[~results_df['fna_exists']]['fna_file'].unique()[:10]:
                        print(f" - {fna}")
                        if fna_missing > 10:
                            print(f" ... more {fna_missing - 10} ")
                            if ann_missing > 0:
                                print(f"\n annotation file:")
                                for _, row in results_df[~results_df['annotation_exists']].head(10).iterrows():
                                    print(f" - {row['renamed_file']} ({row['annotation_type']})")
                                    if ann_missing > 10:
                                        print(f" ... more {ann_missing - 10} ")
                                        # formatmatchedstatistics(onlystatisticsexistsfile)
                                        existing_files = results_df[results_df['annotation_exists']]
                                        if len(existing_files) > 0:
                                            format_mismatch = (~existing_files['format_match']).sum()
                                            print(f"\nformat ( statisticsexistsfile):")
                                            print(f" formatmatched: {existing_files['format_match'].sum()} ")
                                            print(f" format matched: {format_mismatch} ")
                                            if format_mismatch > 0:
                                                print(f"\n format matched file:")
                                                mismatch_files = existing_files[~existing_files['format_match']]
                                                for _, row in mismatch_files.head(20).iterrows():
                                                    print(f" - {row['renamed_file']}")
                                                    print(f" Summary: {row['expected_format']}, to: {row['detected_format']}")
                                                    print(f" GTFSummary: {row['gtf_score']}, GFFSummary: {row['gff_score']}")
                                                    if format_mismatch > 20:
                                                        print(f" ... more {format_mismatch - 20} ")
                                                        # statistics
                                                        print(f"\nGTFfileSummary:")
                                                        gtf_results = results_df[results_df['annotation_type'] == 'gtf']
                                                        gtf_exists = gtf_results['annotation_exists'].sum()
                                                        gtf_format_match = gtf_results[gtf_results['annotation_exists']]['format_match'].sum()
                                                        print(f" file exists: {gtf_exists}/{len(gtf_results)}")
                                                        if gtf_exists > 0:
                                                            print(f" formatcorrect: {gtf_format_match}/{gtf_exists}")
                                                            print(f"\nGFFfileSummary:")
                                                            gff_results = results_df[results_df['annotation_type'] == 'gff']
                                                            gff_exists = gff_results['annotation_exists'].sum()
                                                            gff_format_match = gff_results[gff_results['annotation_exists']]['format_match'].sum()
                                                            print(f" file exists: {gff_exists}/{len(gff_results)}")
                                                            if gff_exists > 0:
                                                                print(f" formatcorrect: {gff_format_match}/{gff_exists}")
                                                                # save results
                                                                output_file = ANNOTATION_DIR / "file_validation_report.tsv"
                                                                results_df.to_csv(output_file, sep='\t', index=False)
                                                                print(f"\ndetailed reportsaved: {output_file}")
                                                                return results_df


def check_naming_patterns():
    """check file """
    print("\n" + "=" * 80)
    print("file analyze")
    print("=" * 80)
    mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
    # GTFfile
    print("\nGTFfileSummary:")
    gtf_files = mapping_df[mapping_df['annotation_type'] == 'gtf']['original_annotation_file']
    gtf_patterns = defaultdict(int)
    for filename in gtf_files:
        # extract
        if '.gtf.gz' in filename:
            pattern = '*.gtf.gz'
        elif '.gtf' in filename:
            pattern = '*.gtf'
        else:
            pattern = 'other'
            gtf_patterns[pattern] += 1
            for pattern, count in sorted(gtf_patterns.items()):
                print(f" {pattern}: {count} ")
                # Processing section
                print(f"\n filename:")
                for filename in gtf_files.head(5):
                    print(f" - {filename}")
                    # GFFfile
                    print("\nGFFfileSummary:")
                    gff_files = mapping_df[mapping_df['annotation_type'] == 'gff']['original_annotation_file']
                    gff_patterns = defaultdict(int)
                    for filename in gff_files:
                        # extract
                        if '.gff.gz' in filename:
                            pattern = '*.gff.gz'
                        elif '.gff3.gz' in filename:
                            pattern = '*.gff3.gz'
                        elif '.gff' in filename:
                            pattern = '*.gff'
                        elif '.gff3' in filename:
                            pattern = '*.gff3'
                        else:
                            pattern = 'other'
                            gff_patterns[pattern] += 1
                            for pattern, count in sorted(gff_patterns.items()):
                                print(f" {pattern}: {count} ")
                                # Processing section
                                print(f"\n filename:")
                                for filename in gff_files.head(5):
                                    print(f" - {filename}")


def main():
    print("=" * 80)
    print("GTF/GFFannotationfile ")
    print("check file path, format and ")
    print("=" * 80)
    # 1. check file
    check_file_extensions()
    # 2. check
    check_naming_patterns()
    # 3. mapping file
    results_df = validate_mapping_files()
    # Processing section
    print("\n" + "=" * 80)
    print(" completed")
    print("=" * 80)
    all_ok = (
        results_df['fna_exists'].all() and results_df['annotation_exists'].all() and results_df[results_df['annotation_exists']]['format_match'].all()
)
if all_ok:
    print("\nOK allfile does not exist formatcorrect! rowsextract .")
else:
    print("\nWarning, detailed information.")
    print("Summary: ")
    print(" 1. check file correctdirectory")
    print(" 2. format matched file correct ")
    print(" 3. file_validation_report.tsv detailed information")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
fromGTF/GFFannotationfileinextractproteinIDandgene information
 GTFandGFFformat, process and
"""

import os
import gzip
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from multiprocessing import Pool
from tqdm import tqdm

# Path configuration
BASE_DIR = Path("/path/to/project/data")
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
MAPPING_FILE = ANNOTATION_DIR / "report/fna_annotation_mapping.tsv"
GTF_OUTPUT_DIR = ANNOTATION_DIR / "gtf_mic"
GFF_OUTPUT_DIR = ANNOTATION_DIR / "gff_mic"
OUTPUT_PROTEIN_INFO = ANNOTATION_DIR / "report/protein_gene_mapping.tsv"
LOG_FILE = ANNOTATION_DIR / "extraction_log.txt"

NUM_CORES = 64


def parse_gtf_attributes(attr_string):
    """
    Parse the GTF-format attributes column
    format: key "value"; key "value";
    Parse strictly according to the GTF standard
    """
    attributes = {}
    # GTFformat: key "value"
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes


def parse_gff_attributes(attr_string):
    """
    Parse the GFF-format attributes column
    format: key=value;key=value
    Parse strictly according to the GFF3 standard
    """
    attributes = {}
    # GFFformat: key=value
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes


def extract_protein_id_gtf(attributes):
    """
    Extract the protein ID from GTF attributes
    Priority: protein_id (standard field)
    """
    protein_id = attributes.get('protein_id', None)
    return protein_id


def extract_gene_name_gtf(attributes):
    """
    Extract the gene name from GTF attributes
    Priority: gene > gene_id > locus_tag
    """
    if 'gene' in attributes:
        return attributes['gene']
    elif 'gene_id' in attributes:
        return attributes['gene_id']
    elif 'locus_tag' in attributes:
        return attributes['locus_tag']
    return None


def extract_protein_id_gff(attributes):
    """
    fromGFF attributesinextractproteinID
    Summary: protein_id > Name > ID(removecds-first )
    Summary: protein_idprotein ID
    ID feature ID( cds-WP_xxx), Name gene orproteinID
    """
    if 'protein_id' in attributes:
        return attributes['protein_id']
    elif 'Name' in attributes:
        name = attributes['Name']
        # Nameprotein ID(or.),
        if '_' in name or '.' in name:
            return name
        # after fromIDinextract
        if 'ID' in attributes:
            protein_id = attributes['ID']
            # ID cds-WP_010229278.1 format
            if protein_id.startswith('cds-'):
                return protein_id[4:] # remove 'cds-' first
            elif protein_id.startswith('cds_'):
                return protein_id[4:] # remove 'cds_' first
            # IDprotein ID, return
            if '_' in protein_id or '.' in protein_id:
                return protein_id
            return None


def extract_gene_name_gff(attributes):
    """
    fromGFF attributesinextractgene
    Summary: gene > locus_tag
    Summary: Name, forNameprotein ID
    """
    if 'gene' in attributes:
        return attributes['gene']
    elif 'locus_tag' in attributes:
        return attributes['locus_tag']
    return None


def validate_gtf_format(line):
    """ forwith GTFformatrows"""
    if line.startswith('#'):
        return True
    parts = line.strip().split('\t')
    if len(parts) < 9:
        return False
    # GTFattributes
    return '"' in parts[8]


def validate_gff_format(line):
    """ forwith GFFformatrows"""
    if line.startswith('#'):
        return True
    parts = line.strip().split('\t')
    if len(parts) < 9:
        return False
    # GFFattributes
    return '=' in parts[8]


def process_gtf_file(gtf_path, fna_seq_ids):
    """
    process GTFfile, extractproteinIDandgene
    onlyextractFNAfileinexistssequenceID information
    """
    results = []
    format_warnings = []
    try:
        if str(gtf_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            line_count = 0
            cds_count = 0
            matched_cds_count = 0
            with open_func(gtf_path, mode) as f:
                for line in f:
                    line_count += 1
                    # skipannotationrows
                    if line.startswith('#'):
                        continue
                    # GTFformat
                    if line_count <= 10 and not validate_gtf_format(line):
                        format_warnings.append(f"Line {line_count} may not be valid GTF format")
                        parts = line.strip().split('\t')
                        if len(parts) < 9:
                            continue
                        seq_id = parts[0]
                        feature = parts[2]
                        attributes_str = parts[8]
                        # statisticsCDS
                        if feature == 'CDS':
                            cds_count += 1
                            # onlyprocessCDS, sequenceID FNAinexists
                            if feature == 'CDS' and seq_id in fna_seq_ids:
                                matched_cds_count += 1
                                attributes = parse_gtf_attributes(attributes_str)
                                protein_id = extract_protein_id_gtf(attributes)
                                gene_name = extract_gene_name_gtf(attributes)
                                if protein_id: # onlyrecordwithproteinID
                                    results.append({
                                        'seq_id': seq_id,
                                        'protein_id': protein_id,
                                        'gene_name': gene_name if gene_name else 'N/A',
                                        'start': int(parts[3]),
                                        'end': int(parts[4]),
                                        'strand': parts[6],
                                        'source': parts[1],
                                        'product': attributes.get('product', 'N/A')
                                    })
    except Exception as e:
        format_warnings.append(f"Error processing file: {e}")
        # returnresultsandstatistics
        stats = {
            'total_lines': line_count,
            'total_cds': cds_count,
            'matched_cds': matched_cds_count,
            'extracted_proteins': len(results),
            'warnings': format_warnings
        }
        return results, stats


def process_gff_file(gff_path, fna_seq_ids):
    """
    process GFFfile, extractproteinIDandgene
    onlyextractFNAfileinexistssequenceID information
    """
    results = []
    format_warnings = []
    try:
        if str(gff_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            line_count = 0
            cds_count = 0
            matched_cds_count = 0
            with open_func(gff_path, mode) as f:
                for line in f:
                    line_count += 1
                    # skipannotationrows
                    if line.startswith('#'):
                        continue
                    # GFFformat
                    if line_count <= 10 and not validate_gff_format(line):
                        format_warnings.append(f"Line {line_count} may not be valid GFF format")
                        parts = line.strip().split('\t')
                        if len(parts) < 9:
                            continue
                        seq_id = parts[0]
                        feature_type = parts[2]
                        attributes_str = parts[8]
                        # statisticsCDS
                        if feature_type == 'CDS':
                            cds_count += 1
                            # onlyprocessCDS, sequenceID FNAinexists
                            if feature_type == 'CDS' and seq_id in fna_seq_ids:
                                matched_cds_count += 1
                                attributes = parse_gff_attributes(attributes_str)
                                protein_id = extract_protein_id_gff(attributes)
                                gene_name = extract_gene_name_gff(attributes)
                                if protein_id: # onlyrecordwithproteinID
                                    results.append({
                                        'seq_id': seq_id,
                                        'protein_id': protein_id,
                                        'gene_name': gene_name if gene_name else 'N/A',
                                        'start': int(parts[3]),
                                        'end': int(parts[4]),
                                        'strand': parts[6],
                                        'source': parts[1],
                                        'product': attributes.get('product', 'N/A')
                                    })
    except Exception as e:
        format_warnings.append(f"Error processing file: {e}")
        # returnresultsandstatistics
        stats = {
            'total_lines': line_count,
            'total_cds': cds_count,
            'matched_cds': matched_cds_count,
            'extracted_proteins': len(results),
            'warnings': format_warnings
        }
        return results, stats


def extract_seq_ids_from_fna(fna_path):
    """fromFNAfileinextractsequenceID """
    seq_ids = set()
    try:
        if str(fna_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(fna_path, mode) as f:
                for line in f:
                    if line.startswith('>'):
                        seq_id = line[1:].split()[0]
                        seq_ids.add(seq_id)
    except Exception as e:
        print(f"Error reading {fna_path}: {e}")
        return seq_ids


def process_mapping_entry(args):
    """
    process mapping
    return FNAfile allproteininformation
     GTFandGFFprocesspath
    """
    row_dict, base_dir = args
    fna_file = row_dict['fna_file']
    species_name = row_dict['species_name']
    annotation_type = row_dict['annotation_type']
    annotation_file = row_dict['original_annotation_file']
    # buildFNAfilepath
    fna_path = base_dir / "reference" / fna_file
    # annotation build file path
    if annotation_type == 'gtf':
        # GTFfile gtf_micdirectory
        renamed_file = f"{species_name}_{annotation_file}"
        annotation_path = base_dir / "genome_annotation" / "gtf_mic" / renamed_file
        processor_name = "GTF"
    elif annotation_type == 'gff':
        # GFFfile gff_micdirectory
        renamed_file = f"{species_name}_{annotation_file}"
        annotation_path = base_dir / "genome_annotation" / "gff_mic" / renamed_file
        processor_name = "GFF"
    else:
        return [], {
        'error': f"Unknown annotation type: {annotation_type}",
        'fna_file': fna_file,
        'annotation_type': annotation_type
    }
    # check file does not exist
    if not fna_path.exists():
        return [], {
        'error': f"FNA file not found: {fna_path}",
        'fna_file': fna_file,
        'annotation_type': annotation_type
    }
    if not annotation_path.exists():
        return [], {
        'error': f"{processor_name} file not found: {annotation_path}",
        'fna_file': fna_file,
        'annotation_type': annotation_type,
        'annotation_file': annotation_file
    }
    # extractFNAinsequenceID
    fna_seq_ids = extract_seq_ids_from_fna(fna_path)
    # process
    if annotation_type == 'gtf':
        results, stats = process_gtf_file(annotation_path, fna_seq_ids)
    else: # gff
        results, stats = process_gff_file(annotation_path, fna_seq_ids)
        # FNAfileandspeciesinformation
        for result in results:
            result['fna_file'] = fna_file
            result['species_name'] = species_name
            result['annotation_type'] = annotation_type #
            result['annotation_file'] = annotation_file
            # process statistics
            stats['fna_file'] = fna_file
            stats['species_name'] = species_name
            stats['annotation_type'] = annotation_type
            stats['annotation_file'] = annotation_file
            stats['fna_seq_count'] = len(fna_seq_ids)
            return results, stats


def main():
    print("=" * 80)
    print("fromGTF/GFFannotationfileinextractproteinIDandgene information")
    print(" format, process and ")
    print("=" * 80)
    # log files
    log_file = open(LOG_FILE, 'w')
    # 1. read mapping file
    print("\n[1/3] readFNA-annotation file mapping ...")
    mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
    print(f" found {len(mapping_df)} mapping ")
    # statisticsGTFandGFFcount
    gtf_count = len(mapping_df[mapping_df['annotation_type'] == 'gtf'])
    gff_count = len(mapping_df[mapping_df['annotation_type'] == 'gff'])
    print(f" in GTF: {gtf_count}, GFF: {gff_count} ")
    log_file.write(f"Total mappings: {len(mapping_df)}\n")
    log_file.write(f"GTF files: {gtf_count}\n")
    log_file.write(f"GFF files: {gff_count}\n\n")
    # 2. rows process all mapping
    print(f"\n[2/3] {NUM_CORES} rowsextractproteininformation...")
    print(f" GTFandGFF process")
    # Processing section
    process_args = [
        (row.to_dict(), BASE_DIR) for _, row in mapping_df.iterrows()
    ]
    # rows process
    all_results = []
    all_stats = []
    error_count = 0
    with Pool(NUM_CORES) as pool:
        for results, stats in tqdm(
            pool.imap_unordered(process_mapping_entry, process_args),
            total=len(process_args),
            desc=" extract "
):
all_results.extend(results)
all_stats.append(stats)
# recorderror
if 'error' in stats:
    error_count += 1
    log_file.write(f"ERROR: {stats['error']}\n")
    log_file.write(f" FNA: {stats['fna_file']}\n")
    log_file.write(f" Type: {stats['annotation_type']}\n\n")
    # recordwarning
    if 'warnings' in stats and stats['warnings']:
        for warning in stats['warnings']:
            log_file.write(f"WARNING [{stats['annotation_type']}]: {warning}\n")
            log_file.write(f" File: {stats['annotation_file']}\n\n")
            print(f" extractto {len(all_results)} proteinrecord")
            if error_count > 0:
                print(f" Warning {error_count} filesprocess, Summary: {LOG_FILE}")
                # 3. saveresultsandstatistics
                print("\n[3/3] saveresults...")
                if not all_results:
                    print(" warning: withextractto proteininformation!")
                    log_file.close()
                    return
                # forDataFrame
                results_df = pd.DataFrame(all_results)
                # column column
                column_order = [
                    'fna_file', 'species_name', 'annotation_type', 'seq_id', 'protein_id', 'gene_name', 'product',
                    'start', 'end', 'strand', 'source', 'annotation_file'
                ]
                results_df = results_df[column_order]
                # saveresults
                results_df.to_csv(OUTPUT_PROTEIN_INFO, sep='\t', index=False)
                # saveprocessstatistics
                stats_df = pd.DataFrame([s for s in all_stats if 'error' not in s])
                stats_output = ANNOTATION_DIR / "extraction_statistics.tsv"
                stats_df.to_csv(stats_output, sep='\t', index=False)
                print(f"\n{'=' * 80}")
                print("extractcompleted!")
                print(f"{'=' * 80}")
                print(f"\nresultsfile: {OUTPUT_PROTEIN_INFO}")
                print(f"statisticsfile: {stats_output}")
                print(f"log files: {LOG_FILE}")
                # statistics
                print(f"\nOverall statistics:")
                print(f" - proteinSummary: {len(results_df)}")
                print(f" - uniqueproteinIDSummary: {results_df['protein_id'].nunique()}")
                print(f" - unique geneSummary: {results_df[results_df['gene_name'] != 'N/A']['gene_name'].nunique()}")
                print(f" - sequenceIDSummary: {results_df['seq_id'].nunique()}")
                print(f" - FNAfileSummary: {results_df['fna_file'].nunique()}")
                # annotation statistics
                print(f"\n annotation statistics:")
                for ann_type in ['gtf', 'gff']:
                    subset = results_df[results_df['annotation_type'] == ann_type]
                    subset_stats = stats_df[stats_df['annotation_type'] == ann_type]
                    if len(subset) > 0:
                        print(f"\n {ann_type.upper()}:")
                        print(f" - fileSummary: {subset['fna_file'].nunique()}")
                        print(f" - proteinrecordSummary: {len(subset)}")
                        print(f" - uniqueproteinID: {subset['protein_id'].nunique()}")
                        print(f" - with geneSummary: {(subset['gene_name'] != 'N/A').sum()} ({(subset['gene_name'] != 'N/A').sum()/len(subset)*100:.1f}%)")
                        if len(subset_stats) > 0:
                            avg_cds = subset_stats['total_cds'].mean()
                            avg_matched = subset_stats['matched_cds'].mean()
                            avg_extracted = subset_stats['extracted_proteins'].mean()
                            print(f" - average files:")
                            print(f" - CDSSummary: {avg_cds:.1f}")
                            print(f" - matchedFNACDS: {avg_matched:.1f}")
                            print(f" - extractprotein: {avg_extracted:.1f}")
                            # format
                            total_warnings = sum(len(s.get('warnings', [])) for s in all_stats)
                            if total_warnings > 0:
                                print(f"\nWarning {total_warnings} format warning, log files")
                                print(f"\nNote: Summary:")
                                print(f" - GTFandGFF process")
                                print(f" - records annotation_type ")
                                print(f" - passedannotation_type data ")
                                print(f" - extraction_statistics.tsv filesprocess ")
                                log_file.close()


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Extract WP_ protein IDs from annotation files and map them to chromosome sequence IDs
statisticsWP_andNP_protein
"""

import os
import gzip
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from multiprocessing import Pool
from tqdm import tqdm

# Path configuration
BASE_DIR = Path("/path/to/project/data")
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
GTF_OUTPUT_DIR = ANNOTATION_DIR / "gtf_mic"
GFF_OUTPUT_DIR = ANNOTATION_DIR / "gff_mic"
MAPPING_FILE = ANNOTATION_DIR / "report/fna_annotation_mapping.tsv"

# Output files
OUTPUT_WP_MAPPING = ANNOTATION_DIR / "chromosome_wp_protein_mapping.tsv"
OUTPUT_SPECIES_STATS = ANNOTATION_DIR / "species_protein_statistics.tsv"
OUTPUT_NP_ONLY_SPECIES = ANNOTATION_DIR / "np_only_species.tsv"

# Number of parallel cores
NUM_CORES = 64


def parse_gtf_attributes(attr_string):
    """
    Parse the GTF-format attribute string
    GTFformat: key "value"; key "value";
    """
    attributes = {}
    # matched key "value" or key 'value'
    pattern = r'(\w+)\s+"([^"]+)"|(\w+)\s+\'([^\']+)\''
    matches = re.findall(pattern, attr_string)
    for match in matches:
        if match[0]: #
            key, value = match[0], match[1]
        else: #
            key, value = match[2], match[3]
            attributes[key] = value
            return attributes


def parse_gff_attributes(attr_string):
    """
    Parse the GFF-format attribute string
    GFFformat: key=value;key=value;
    """
    attributes = {}
    # key=value
    pairs = attr_string.strip().split(';')
    for pair in pairs:
        pair = pair.strip()
        if not pair or '=' not in pair:
            continue
        key, value = pair.split('=', 1)
        attributes[key.strip()] = value.strip()
        return attributes


def extract_protein_ids_from_annotation(annotation_path, file_type):
    """
    fromannotationfileinextractproteinID sequenceID
    return: {
    'wp_proteins': [(seq_id, protein_id), ...],
    'np_proteins': [(seq_id, protein_id), ...],
    'other_proteins': [(seq_id, protein_id), ...]
    }
    """
    wp_proteins = []
    np_proteins = []
    other_proteins = []
    try:
        # Check whether the file is compressed
        if str(annotation_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(annotation_path, mode) as f:
                for line in f:
                    # skipannotationrows
                    if line.startswith('#'):
                        continue
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split('\t')
                    if len(parts) < 9:
                        continue
                    seq_id = parts[0] # /sequenceID( column)
                    feature_type = parts[2] # ( column)
                    attributes = parts[8] # ( column)
                    # onlyprocessCDS
                    if feature_type != 'CDS':
                        continue
                    # file
                    if file_type == 'gtf':
                        attrs = parse_gtf_attributes(attributes)
                    else: # gff
                        attrs = parse_gff_attributes(attributes)
                        # extractprotein_id
                        protein_id = attrs.get('protein_id', '')
                        if not protein_id:
                            continue
                        # protein ID
                        if protein_id.startswith('WP_'):
                            wp_proteins.append((seq_id, protein_id))
                        elif protein_id.startswith('NP_'):
                            np_proteins.append((seq_id, protein_id))
                        else:
                            other_proteins.append((seq_id, protein_id))
    except Exception as e:
        print(f"Error reading {annotation_path}: {e}")
        return {
        'wp_proteins': wp_proteins,
        'np_proteins': np_proteins,
        'other_proteins': other_proteins
    }


def process_annotation_file(args):
    """process annotation file"""
    row_dict, gtf_output_dir, gff_output_dir = args
    species_name = row_dict['species_name']
    fna_file = row_dict['fna_file']
    annotation_type = row_dict['annotation_type']
    original_file = row_dict['original_annotation_file']
    # buildannotationfile path
    if annotation_type == 'gtf':
        # afterfilename
        new_filename = f"{species_name}_{original_file}"
        annotation_path = gtf_output_dir / new_filename
    else: # gff
        new_filename = f"{species_name}_{original_file}"
        annotation_path = gff_output_dir / new_filename
        # check file does not exist
        if not annotation_path.exists():
            print(f"Warning: File not found: {annotation_path}")
            return None
        # extractproteinID
        protein_data = extract_protein_ids_from_annotation(annotation_path, annotation_type)
        # statistics
        wp_count = len(protein_data['wp_proteins'])
        np_count = len(protein_data['np_proteins'])
        other_count = len(protein_data['other_proteins'])
        result = {
            'species_name': species_name,
            'fna_file': fna_file,
            'annotation_file': annotation_path.name,
            'annotation_type': annotation_type,
            'wp_count': wp_count,
            'np_count': np_count,
            'other_count': other_count,
            'total_proteins': wp_count + np_count + other_count,
            'has_wp': wp_count > 0,
            'has_np': np_count > 0,
            'wp_proteins': protein_data['wp_proteins'],
            'np_proteins': protein_data['np_proteins'],
            'other_proteins': protein_data['other_proteins']
        }
        return result


def main():
    print("=" * 80)
    print("fromannotationfileinextractWP_proteinID ")
    print("=" * 80)
    # 1. readmappingfile
    print("\n[1/4] readfna annotation file mapping ...")
    if not MAPPING_FILE.exists():
        print(f"error: mapping file does not exist: {MAPPING_FILE}")
        return
    mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
    print(f" load {len(mapping_df)} mapping ")
    # 2. rows process all annotation file
    print(f"\n[2/4] {NUM_CORES} rowsextractproteinID...")
    # Processing section
    process_args = [
        (row.to_dict(), GTF_OUTPUT_DIR, GFF_OUTPUT_DIR)
        for _, row in mapping_df.iterrows()
    ]
    # rows process
    all_results = []
    with Pool(NUM_CORES) as pool:
        for result in tqdm(pool.imap_unordered(process_annotation_file, process_args),
            total=len(process_args), desc=" extract "):
        if result:
            all_results.append(result)
            print(f" processed successfully {len(all_results)} annotationfile")
            if not all_results:
                print("error: withsuccessextract proteininformation!")
                return
            # 3. saveWP_protein mapping
            print("\n[3/4] WP_protein mapping ...")
            wp_mapping_records = []
            for result in all_results:
                species_name = result['species_name']
                fna_file = result['fna_file']
                annotation_file = result['annotation_file']
                for seq_id, protein_id in result['wp_proteins']:
                    wp_mapping_records.append({
                        'species_name': species_name,
                        'fna_file': fna_file,
                        'annotation_file': annotation_file,
                        'chromosome_seq_id': seq_id,
                        'wp_protein_id': protein_id
                    })
                    # saveWP_proteinmapping
                    if wp_mapping_records:
                        wp_mapping_df = pd.DataFrame(wp_mapping_records)
                        wp_mapping_df.to_csv(OUTPUT_WP_MAPPING, sep='\t', index=False)
                        print(f" save {len(wp_mapping_records)} WP_proteinmapping to: {OUTPUT_WP_MAPPING}")
                    else:
                        print(" warning: with found WP_protein!")
                        # 4. statistics save species proteinstatistics
                        print("\n[4/4] generatespeciesproteinstatistics...")
                        # createstatisticsDataFrame
                        stats_records = []
                        for result in all_results:
                            stats_records.append({
                                'species_name': result['species_name'],
                                'fna_file': result['fna_file'],
                                'annotation_file': result['annotation_file'],
                                'annotation_type': result['annotation_type'],
                                'wp_count': result['wp_count'],
                                'np_count': result['np_count'],
                                'other_count': result['other_count'],
                                'total_proteins': result['total_proteins'],
                                'has_wp': result['has_wp'],
                                'has_np': result['has_np'],
                                'wp_percentage': result['wp_count'] / result['total_proteins'] * 100 if result['total_proteins'] > 0 else 0,
                                'np_percentage': result['np_count'] / result['total_proteins'] * 100 if result['total_proteins'] > 0 else 0
                            })
                            stats_df = pd.DataFrame(stats_records)
                            stats_df = stats_df.sort_values('species_name')
                            stats_df.to_csv(OUTPUT_SPECIES_STATS, sep='\t', index=False)
                            print(f" savespeciesstatisticsto: {OUTPUT_SPECIES_STATS}")
                            # onlywithNP_ withWP_species
                            np_only_df = stats_df[(stats_df['has_np'] == True) & (stats_df['has_wp'] == False)]
                            if len(np_only_df) > 0:
                                np_only_df.to_csv(OUTPUT_NP_ONLY_SPECIES, sep='\t', index=False)
                                print(f" saveonlywithNP_speciesinformationto: {OUTPUT_NP_ONLY_SPECIES}")
                                # 5. statistics
                                print(f"\n{'=' * 80}")
                                print("Processing completed!statisticsSummary: ")
                                print(f"{'=' * 80}")
                                total_species = len(all_results)
                                species_with_wp = stats_df['has_wp'].sum()
                                species_with_np = stats_df['has_np'].sum()
                                species_with_both = ((stats_df['has_wp'] == True) & (stats_df['has_np'] == True)).sum()
                                species_np_only = len(np_only_df)
                                species_wp_only = ((stats_df['has_wp'] == True) & (stats_df['has_np'] == False)).sum()
                                print(f"\nspeciesstatistics:")
                                print(f" - speciesSummary: {total_species}")
                                print(f" - withWP_proteinspecies: {species_with_wp} ({species_with_wp/total_species*100:.1f}%)")
                                print(f" - withNP_proteinspecies: {species_with_np} ({species_with_np/total_species*100:.1f}%)")
                                print(f" - withWP_andNP_species: {species_with_both} ({species_with_both/total_species*100:.1f}%)")
                                print(f" - onlywithWP_species: {species_wp_only} ({species_wp_only/total_species*100:.1f}%)")
                                print(f" - onlywithNP_species: {species_np_only} ({species_np_only/total_species*100:.1f}%)")
                                print(f"\nproteincountstatistics:")
                                total_wp = stats_df['wp_count'].sum()
                                total_np = stats_df['np_count'].sum()
                                total_other = stats_df['other_count'].sum()
                                total_all = stats_df['total_proteins'].sum()
                                print(f" - WP_proteinSummary: {total_wp:,} ({total_wp/total_all*100:.1f}%)")
                                print(f" - NP_proteinSummary: {total_np:,} ({total_np/total_all*100:.1f}%)")
                                print(f" - proteinSummary: {total_other:,} ({total_other/total_all*100:.1f}%)")
                                print(f" - proteinSummary: {total_all:,}")
                                print(f"\naverage species:")
                                print(f" - averageWP_proteinSummary: {stats_df['wp_count'].mean():.1f}")
                                print(f" - averageNP_proteinSummary: {stats_df['np_count'].mean():.1f}")
                                print(f" - average proteinSummary: {stats_df['total_proteins'].mean():.1f}")
                                if species_np_only > 0:
                                    print(f"\nWarning {species_np_only} speciesonlywithNP_protein, withWP_protein!")
                                    print(f" detailed informationSummary: {OUTPUT_NP_ONLY_SPECIES}")
                                    print("\n speciesSummary:")
                                    for species in np_only_df['species_name'].head(10):
                                        print(f" - {species}")
                                        if len(np_only_df) > 10:
                                            print(f" ... more {len(np_only_df) - 10} ")
                                            print(f"\nOutput files:")
                                            print(f" - WP_proteinmapping: {OUTPUT_WP_MAPPING}")
                                            print(f" - speciesstatistics: {OUTPUT_SPECIES_STATS}")
                                            if species_np_only > 0:
                                                print(f" - onlywithNP_species: {OUTPUT_NP_ONLY_SPECIES}")
                                                print(f"\nNote: Summary:")
                                                print(f" - {OUTPUT_WP_MAPPING} WP_protein ")
                                                print(f" - {OUTPUT_SPECIES_STATS} speciesprotein ")
                                                print(f" - WP_protein RefSeqprotein ID, species ")
                                                print(f" - NP_protein geneprotein ID")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Extract WP_ protein IDs from annotation files and map them to chromosome sequence IDs
Summarize WP_ proteins in GTF and GFF files separately and save detailed lists
"""

import os
import gzip
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from multiprocessing import Pool
from tqdm import tqdm

# Path configuration
BASE_DIR = Path("/path/to/project/data")
ANNOTATION_DIR = BASE_DIR / "genome_annotation"
GTF_OUTPUT_DIR = ANNOTATION_DIR / "gtf_mic"
GFF_OUTPUT_DIR = ANNOTATION_DIR / "gff_mic"
MAPPING_FILE = ANNOTATION_DIR / "report/fna_annotation_mapping.tsv"

# Output files - saveGTFandGFFresults
OUTPUT_GTF_WP_LIST = ANNOTATION_DIR / "report2/gtf_wp_protein_list.tsv"
OUTPUT_GFF_WP_LIST = ANNOTATION_DIR / "report2/gff_wp_protein_list.tsv"
OUTPUT_GTF_WP_MAPPING = ANNOTATION_DIR / "report2/gtf_chromosome_wp_mapping.tsv"
OUTPUT_GFF_WP_MAPPING = ANNOTATION_DIR / "report2/gff_chromosome_wp_mapping.tsv"
OUTPUT_GTF_STATS = ANNOTATION_DIR / "report2/gtf_protein_statistics.tsv"
OUTPUT_GFF_STATS = ANNOTATION_DIR / "report2/gff_protein_statistics.tsv"
OUTPUT_NP_ONLY_SPECIES = ANNOTATION_DIR / "report2/np_only_species.tsv"
OUTPUT_COMPARISON = ANNOTATION_DIR / "report2/gtf_vs_gff_comparison.tsv"

# Number of parallel cores
NUM_CORES = 64


def parse_gtf_attributes(attr_string):
    """
    Parse the GTF-format attribute string
    GTFformat: key "value"; key "value";
    """
    attributes = {}
    # matched key "value" or key 'value'
    pattern = r'(\w+)\s+"([^"]+)"|(\w+)\s+\'([^\']+)\''
    matches = re.findall(pattern, attr_string)
    for match in matches:
        if match[0]: #
            key, value = match[0], match[1]
        else: #
            key, value = match[2], match[3]
            attributes[key] = value
            return attributes


def parse_gff_attributes(attr_string):
    """
    Parse the GFF-format attribute string
    GFFformat: key=value;key=value;
    """
    attributes = {}
    # key=value
    pairs = attr_string.strip().split(';')
    for pair in pairs:
        pair = pair.strip()
        if not pair or '=' not in pair:
            continue
        key, value = pair.split('=', 1)
        attributes[key.strip()] = value.strip()
        return attributes


def extract_protein_ids_from_annotation(annotation_path, file_type):
    """
    fromannotationfileinextractproteinID sequenceID
    return: {
    'wp_proteins': [(seq_id, protein_id), ...],
    'np_proteins': [(seq_id, protein_id), ...],
    'other_proteins': [(seq_id, protein_id), ...]
    }
    """
    wp_proteins = []
    np_proteins = []
    other_proteins = []
    try:
        # Check whether the file is compressed
        if str(annotation_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(annotation_path, mode) as f:
                for line in f:
                    # skipannotationrows
                    if line.startswith('#'):
                        continue
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split('\t')
                    if len(parts) < 9:
                        continue
                    seq_id = parts[0] # /sequenceID( column)
                    feature_type = parts[2] # ( column)
                    attributes = parts[8] # ( column)
                    # onlyprocessCDS
                    if feature_type != 'CDS':
                        continue
                    # file
                    if file_type == 'gtf':
                        attrs = parse_gtf_attributes(attributes)
                    else: # gff
                        attrs = parse_gff_attributes(attributes)
                        # extractprotein_id
                        protein_id = attrs.get('protein_id', '')
                        if not protein_id:
                            continue
                        # protein ID
                        if protein_id.startswith('WP_'):
                            wp_proteins.append((seq_id, protein_id))
                        elif protein_id.startswith('NP_'):
                            np_proteins.append((seq_id, protein_id))
                        else:
                            other_proteins.append((seq_id, protein_id))
    except Exception as e:
        print(f"Error reading {annotation_path}: {e}")
        return {
        'wp_proteins': wp_proteins,
        'np_proteins': np_proteins,
        'other_proteins': other_proteins
    }


def process_annotation_file(args):
    """process annotation file"""
    row_dict, gtf_output_dir, gff_output_dir = args
    species_name = row_dict['species_name']
    fna_file = row_dict['fna_file']
    annotation_type = row_dict['annotation_type']
    original_file = row_dict['original_annotation_file']
    # buildannotationfile path
    if annotation_type == 'gtf':
        # afterfilename
        new_filename = f"{species_name}_{original_file}"
        annotation_path = gtf_output_dir / new_filename
    else: # gff
        new_filename = f"{species_name}_{original_file}"
        annotation_path = gff_output_dir / new_filename
        # check file does not exist
        if not annotation_path.exists():
            print(f"Warning: File not found: {annotation_path}")
            return None
        # extractproteinID
        protein_data = extract_protein_ids_from_annotation(annotation_path, annotation_type)
        # uniqueWP_proteinID(deduplicate)
        unique_wp = set([pid for _, pid in protein_data['wp_proteins']])
        unique_np = set([pid for _, pid in protein_data['np_proteins']])
        unique_other = set([pid for _, pid in protein_data['other_proteins']])
        # statistics
        wp_count = len(unique_wp)
        np_count = len(unique_np)
        other_count = len(unique_other)
        total_wp_entries = len(protein_data['wp_proteins']) #
        result = {
            'species_name': species_name,
            'fna_file': fna_file,
            'annotation_file': annotation_path.name,
            'annotation_type': annotation_type,
            'unique_wp_count': wp_count,
            'total_wp_entries': total_wp_entries, # WP_ annotationfilein
            'unique_np_count': np_count,
            'unique_other_count': other_count,
            'total_unique_proteins': wp_count + np_count + other_count,
            'has_wp': wp_count > 0,
            'has_np': np_count > 0,
            'wp_protein_list': sorted(list(unique_wp)), # uniqueWP_list
            'np_protein_list': sorted(list(unique_np)),
            'wp_proteins_with_chr': protein_data['wp_proteins'], # information
            'np_proteins_with_chr': protein_data['np_proteins'],
            'other_proteins_with_chr': protein_data['other_proteins']
        }
        return result


def main():
    print("=" * 80)
    print("fromannotationfileinextractWP_proteinID - statisticsGTFandGFF")
    print("=" * 80)
    # 1. readmappingfile
    print("\n[1/5] readfna annotation file mapping ...")
    if not MAPPING_FILE.exists():
        print(f"error: mapping file does not exist: {MAPPING_FILE}")
        return
    mapping_df = pd.read_csv(MAPPING_FILE, sep='\t')
    print(f" load {len(mapping_df)} mapping ")
    gtf_count = len(mapping_df[mapping_df['annotation_type'] == 'gtf'])
    gff_count = len(mapping_df[mapping_df['annotation_type'] == 'gff'])
    print(f" in GTF: {gtf_count}, GFF: {gff_count} ")
    # 2. rows process all annotation file
    print(f"\n[2/5] {NUM_CORES} rowsextractproteinID...")
    # Processing section
    process_args = [
        (row.to_dict(), GTF_OUTPUT_DIR, GFF_OUTPUT_DIR)
        for _, row in mapping_df.iterrows()
    ]
    # rows process
    all_results = []
    with Pool(NUM_CORES) as pool:
        for result in tqdm(pool.imap_unordered(process_annotation_file, process_args),
            total=len(process_args), desc=" extract "):
        if result:
            all_results.append(result)
            print(f" processed successfully {len(all_results)} annotationfile")
            if not all_results:
                print("error: withsuccessextract proteininformation!")
                return
            # GTFandGFFresults
            gtf_results = [r for r in all_results if r['annotation_type'] == 'gtf']
            gff_results = [r for r in all_results if r['annotation_type'] == 'gff']
            print(f" GTFresults: {len(gtf_results)} ")
            print(f" GFFresults: {len(gff_results)} ")
            # 3. saveGTFWP_proteinlistandmapping
            print("\n[3/5] saveGTFfileWP_proteininformation...")
            if gtf_results:
                # GTF WP_proteinlist( speciesuniqueWP_list)
                gtf_wp_list_records = []
                for result in gtf_results:
                    wp_list_str = ','.join(result['wp_protein_list']) if result['wp_protein_list'] else ''
                    gtf_wp_list_records.append({
                        'species_name': result['species_name'],
                        'fna_file': result['fna_file'],
                        'annotation_file': result['annotation_file'],
                        'unique_wp_count': result['unique_wp_count'],
                        'total_wp_entries': result['total_wp_entries'],
                        'wp_protein_list': wp_list_str
                    })
                    gtf_wp_list_df = pd.DataFrame(gtf_wp_list_records)
                    gtf_wp_list_df = gtf_wp_list_df.sort_values('species_name')
                    gtf_wp_list_df.to_csv(OUTPUT_GTF_WP_LIST, sep='\t', index=False)
                    print(f" OK GTF WP_proteinlist: {OUTPUT_GTF_WP_LIST}")
                    # GTF mapping
                    gtf_wp_mapping_records = []
                    for result in gtf_results:
                        for seq_id, protein_id in result['wp_proteins_with_chr']:
                            gtf_wp_mapping_records.append({
                                'species_name': result['species_name'],
                                'fna_file': result['fna_file'],
                                'annotation_file': result['annotation_file'],
                                'chromosome_seq_id': seq_id,
                                'wp_protein_id': protein_id
                            })
                            if gtf_wp_mapping_records:
                                gtf_wp_mapping_df = pd.DataFrame(gtf_wp_mapping_records)
                                gtf_wp_mapping_df.to_csv(OUTPUT_GTF_WP_MAPPING, sep='\t', index=False)
                                print(f" OK GTF mapping: {OUTPUT_GTF_WP_MAPPING} ({len(gtf_wp_mapping_records):,} )")
                                # GTF statistics
                                gtf_stats_records = []
                                for result in gtf_results:
                                    gtf_stats_records.append({
                                        'species_name': result['species_name'],
                                        'fna_file': result['fna_file'],
                                        'annotation_file': result['annotation_file'],
                                        'unique_wp_count': result['unique_wp_count'],
                                        'total_wp_entries': result['total_wp_entries'],
                                        'unique_np_count': result['unique_np_count'],
                                        'unique_other_count': result['unique_other_count'],
                                        'total_unique_proteins': result['total_unique_proteins'],
                                        'has_wp': result['has_wp'],
                                        'has_np': result['has_np'],
                                        'wp_percentage': result['unique_wp_count'] / result['total_unique_proteins'] * 100 if result['total_unique_proteins'] > 0 else 0,
                                        'np_percentage': result['unique_np_count'] / result['total_unique_proteins'] * 100 if result['total_unique_proteins'] > 0 else 0
                                    })
                                    gtf_stats_df = pd.DataFrame(gtf_stats_records)
                                    gtf_stats_df = gtf_stats_df.sort_values('species_name')
                                    gtf_stats_df.to_csv(OUTPUT_GTF_STATS, sep='\t', index=False)
                                    print(f" OK GTF statistics: {OUTPUT_GTF_STATS}")
                                else:
                                    print(" warning: withGTFresults")
                                    # 4. saveGFFWP_proteinlistandmapping
                                    print("\n[4/5] saveGFFfileWP_proteininformation...")
                                    if gff_results:
                                        # GFF WP_proteinlist( speciesuniqueWP_list)
                                        gff_wp_list_records = []
                                        for result in gff_results:
                                            wp_list_str = ','.join(result['wp_protein_list']) if result['wp_protein_list'] else ''
                                            gff_wp_list_records.append({
                                                'species_name': result['species_name'],
                                                'fna_file': result['fna_file'],
                                                'annotation_file': result['annotation_file'],
                                                'unique_wp_count': result['unique_wp_count'],
                                                'total_wp_entries': result['total_wp_entries'],
                                                'wp_protein_list': wp_list_str
                                            })
                                            gff_wp_list_df = pd.DataFrame(gff_wp_list_records)
                                            gff_wp_list_df = gff_wp_list_df.sort_values('species_name')
                                            gff_wp_list_df.to_csv(OUTPUT_GFF_WP_LIST, sep='\t', index=False)
                                            print(f" OK GFF WP_proteinlist: {OUTPUT_GFF_WP_LIST}")
                                            # GFF mapping
                                            gff_wp_mapping_records = []
                                            for result in gff_results:
                                                for seq_id, protein_id in result['wp_proteins_with_chr']:
                                                    gff_wp_mapping_records.append({
                                                        'species_name': result['species_name'],
                                                        'fna_file': result['fna_file'],
                                                        'annotation_file': result['annotation_file'],
                                                        'chromosome_seq_id': seq_id,
                                                        'wp_protein_id': protein_id
                                                    })
                                                    if gff_wp_mapping_records:
                                                        gff_wp_mapping_df = pd.DataFrame(gff_wp_mapping_records)
                                                        gff_wp_mapping_df.to_csv(OUTPUT_GFF_WP_MAPPING, sep='\t', index=False)
                                                        print(f" OK GFF mapping: {OUTPUT_GFF_WP_MAPPING} ({len(gff_wp_mapping_records):,} )")
                                                        # GFF statistics
                                                        gff_stats_records = []
                                                        for result in gff_results:
                                                            gff_stats_records.append({
                                                                'species_name': result['species_name'],
                                                                'fna_file': result['fna_file'],
                                                                'annotation_file': result['annotation_file'],
                                                                'unique_wp_count': result['unique_wp_count'],
                                                                'total_wp_entries': result['total_wp_entries'],
                                                                'unique_np_count': result['unique_np_count'],
                                                                'unique_other_count': result['unique_other_count'],
                                                                'total_unique_proteins': result['total_unique_proteins'],
                                                                'has_wp': result['has_wp'],
                                                                'has_np': result['has_np'],
                                                                'wp_percentage': result['unique_wp_count'] / result['total_unique_proteins'] * 100 if result['total_unique_proteins'] > 0 else 0,
                                                                'np_percentage': result['unique_np_count'] / result['total_unique_proteins'] * 100 if result['total_unique_proteins'] > 0 else 0
                                                            })
                                                            gff_stats_df = pd.DataFrame(gff_stats_records)
                                                            gff_stats_df = gff_stats_df.sort_values('species_name')
                                                            gff_stats_df.to_csv(OUTPUT_GFF_STATS, sep='\t', index=False)
                                                            print(f" OK GFF statistics: {OUTPUT_GFF_STATS}")
                                                        else:
                                                            print(" warning: withGFFresults")
                                                            # 5. generateanalyzeandonlywithNP_species
                                                            print("\n[5/5] generateanalyze...")
                                                            # withGTFandGFFspecies, rows
                                                            gtf_species = {r['species_name']: r for r in gtf_results}
                                                            gff_species = {r['species_name']: r for r in gff_results}
                                                            common_species = set(gtf_species.keys()) & set(gff_species.keys())
                                                            if common_species:
                                                                comparison_records = []
                                                                for species in sorted(common_species):
                                                                    gtf_r = gtf_species[species]
                                                                    gff_r = gff_species[species]
                                                                    # calculateWP_protein
                                                                    gtf_wp_set = set(gtf_r['wp_protein_list'])
                                                                    gff_wp_set = set(gff_r['wp_protein_list'])
                                                                    common_wp = gtf_wp_set & gff_wp_set
                                                                    gtf_only_wp = gtf_wp_set - gff_wp_set
                                                                    gff_only_wp = gff_wp_set - gtf_wp_set
                                                                    comparison_records.append({
                                                                        'species_name': species,
                                                                        'fna_file': gtf_r['fna_file'],
                                                                        'gtf_file': gtf_r['annotation_file'],
                                                                        'gff_file': gff_r['annotation_file'],
                                                                        'gtf_unique_wp': len(gtf_wp_set),
                                                                        'gff_unique_wp': len(gff_wp_set),
                                                                        'common_wp': len(common_wp),
                                                                        'gtf_only_wp': len(gtf_only_wp),
                                                                        'gff_only_wp': len(gff_only_wp),
                                                                        'wp_overlap_ratio': len(common_wp) / max(len(gtf_wp_set | gff_wp_set), 1),
                                                                        'common_wp_list': ','.join(sorted(common_wp)) if common_wp else '',
                                                                        'gtf_only_wp_list': ','.join(sorted(gtf_only_wp)) if gtf_only_wp else '',
                                                                        'gff_only_wp_list': ','.join(sorted(gff_only_wp)) if gff_only_wp else ''
                                                                    })
                                                                    comparison_df = pd.DataFrame(comparison_records)
                                                                    comparison_df.to_csv(OUTPUT_COMPARISON, sep='\t', index=False)
                                                                    print(f" OK GTF vs GFF Summary: {OUTPUT_COMPARISON} ({len(common_species)} species)")
                                                                    # onlywithNP_ withWP_species( GTFandGFF)
                                                                    np_only_species = []
                                                                    for result in all_results:
                                                                        if result['has_np'] and not result['has_wp']:
                                                                            np_only_species.append({
                                                                                'species_name': result['species_name'],
                                                                                'fna_file': result['fna_file'],
                                                                                'annotation_file': result['annotation_file'],
                                                                                'annotation_type': result['annotation_type'],
                                                                                'unique_np_count': result['unique_np_count'],
                                                                                'unique_other_count': result['unique_other_count']
                                                                            })
                                                                            if np_only_species:
                                                                                np_only_df = pd.DataFrame(np_only_species)
                                                                                np_only_df = np_only_df.sort_values('species_name')
                                                                                np_only_df.to_csv(OUTPUT_NP_ONLY_SPECIES, sep='\t', index=False)
                                                                                print(f" OK onlywithNP_species: {OUTPUT_NP_ONLY_SPECIES} ({len(np_only_species)})")
                                                                                # 6. statistics
                                                                                print(f"\n{'=' * 80}")
                                                                                print("Processing completed!statisticsSummary: ")
                                                                                print(f"{'=' * 80}")
                                                                                # GTFstatistics
                                                                                if gtf_results:
                                                                                    print(f"\nStatistics: GTFfilestatistics:")
                                                                                    gtf_total_species = len(gtf_results)
                                                                                    gtf_with_wp = sum(1 for r in gtf_results if r['has_wp'])
                                                                                    gtf_with_np = sum(1 for r in gtf_results if r['has_np'])
                                                                                    gtf_np_only = sum(1 for r in gtf_results if r['has_np'] and not r['has_wp'])
                                                                                    gtf_total_wp = sum(r['unique_wp_count'] for r in gtf_results)
                                                                                    gtf_total_np = sum(r['unique_np_count'] for r in gtf_results)
                                                                                    gtf_avg_wp = gtf_total_wp / gtf_total_species if gtf_total_species > 0 else 0
                                                                                    print(f" - speciesSummary: {gtf_total_species}")
                                                                                    print(f" - withWP_species: {gtf_with_wp} ({gtf_with_wp/gtf_total_species*100:.1f}%)")
                                                                                    print(f" - withNP_species: {gtf_with_np} ({gtf_with_np/gtf_total_species*100:.1f}%)")
                                                                                    print(f" - onlywithNP_species: {gtf_np_only}")
                                                                                    print(f" - uniqueWP_Summary: {gtf_total_wp:,}")
                                                                                    print(f" - uniqueNP_Summary: {gtf_total_np:,}")
                                                                                    print(f" - average speciesWP_Summary: {gtf_avg_wp:.1f}")
                                                                                    # GFFstatistics
                                                                                    if gff_results:
                                                                                        print(f"\nStatistics: GFFfilestatistics:")
                                                                                        gff_total_species = len(gff_results)
                                                                                        gff_with_wp = sum(1 for r in gff_results if r['has_wp'])
                                                                                        gff_with_np = sum(1 for r in gff_results if r['has_np'])
                                                                                        gff_np_only = sum(1 for r in gff_results if r['has_np'] and not r['has_wp'])
                                                                                        gff_total_wp = sum(r['unique_wp_count'] for r in gff_results)
                                                                                        gff_total_np = sum(r['unique_np_count'] for r in gff_results)
                                                                                        gff_avg_wp = gff_total_wp / gff_total_species if gff_total_species > 0 else 0
                                                                                        print(f" - speciesSummary: {gff_total_species}")
                                                                                        print(f" - withWP_species: {gff_with_wp} ({gff_with_wp/gff_total_species*100:.1f}%)")
                                                                                        print(f" - withNP_species: {gff_with_np} ({gff_with_np/gff_total_species*100:.1f}%)")
                                                                                        print(f" - onlywithNP_species: {gff_np_only}")
                                                                                        print(f" - uniqueWP_Summary: {gff_total_wp:,}")
                                                                                        print(f" - uniqueNP_Summary: {gff_total_np:,}")
                                                                                        print(f" - average speciesWP_Summary: {gff_avg_wp:.1f}")
                                                                                        # statistics
                                                                                        if common_species and comparison_records:
                                                                                            print(f"\nProcessing: GTF vs GFF ({len(common_species)} species):")
                                                                                            avg_overlap = sum(r['wp_overlap_ratio'] for r in comparison_records) / len(comparison_records)
                                                                                            perfect_match = sum(1 for r in comparison_records if r['wp_overlap_ratio'] > 0.99)
                                                                                            print(f" - averageWP_Summary: {avg_overlap:.1%}")
                                                                                            print(f" - matched(>99%): {perfect_match} ")
                                                                                            print(f"\nFiles: Output files:")
                                                                                            print(f"\n GTFSummary:")
                                                                                            if gtf_results:
                                                                                                print(f" - WP_proteinlist: {OUTPUT_GTF_WP_LIST}")
                                                                                                print(f" - mapping: {OUTPUT_GTF_WP_MAPPING}")
                                                                                                print(f" - statistics: {OUTPUT_GTF_STATS}")
                                                                                                print(f"\n GFFSummary:")
                                                                                                if gff_results:
                                                                                                    print(f" - WP_proteinlist: {OUTPUT_GFF_WP_LIST}")
                                                                                                    print(f" - mapping: {OUTPUT_GFF_WP_MAPPING}")
                                                                                                    print(f" - statistics: {OUTPUT_GFF_STATS}")
                                                                                                    print(f"\n Summary:")
                                                                                                    if common_species:
                                                                                                        print(f" - GTF vs GFFSummary: {OUTPUT_COMPARISON}")
                                                                                                        if np_only_species:
                                                                                                            print(f" - onlywithNP_species: {OUTPUT_NP_ONLY_SPECIES}")
                                                                                                            print(f"\nNote: Summary:")
                                                                                                            print(f" - *_wp_protein_list.tsv file species WP_proteinlist")
                                                                                                            print(f" - unique_wp_count deduplicateafteruniqueWP_proteincount")
                                                                                                            print(f" - total_wp_entries WP_ annotationfilein ( )")
                                                                                                            print(f" - *_chromosome_wp_mapping.tsv sequenceID WP_detailed ")


if __name__ == "__main__":
    main()

# Find the corresponding assembly for missing WP IDs

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Use species names from missing_wp_proteins.tsv to find matching annotation files and assembly information
Generate a subset of V7_matched_assembly_info.xlsx
"""

import re
from pathlib import Path
import pandas as pd
from collections import defaultdict

# Path configuration
BASE_DIR = Path("/path/to/project/data/genome_annotation")
MISSING_WP_FILE = BASE_DIR / "report2" / "missing_wp_proteins.tsv"
GTF_DIR = BASE_DIR / "gtf_mic"
GFF_DIR = BASE_DIR / "gff_mic"
ASSEMBLY_INFO = BASE_DIR / "V7_matched_assembly_info.xlsx"
FNA_MAPPING = BASE_DIR / "report/fna_annotation_mapping.tsv"

# Output files
OUTPUT_DIR = BASE_DIR / "report2"
OUTPUT_ASSEMBLY_SUBSET = OUTPUT_DIR / "missing_wp_assembly_info.xlsx"
OUTPUT_SPECIES_ANNOTATION = OUTPUT_DIR / "missing_wp_species_annotation.tsv"
OUTPUT_SUMMARY = OUTPUT_DIR / "missing_wp_summary.tsv"


def extract_gcf_from_filename(filename):
    """Extract the GCF accession from the filename(assembly accession)"""
    match = re.search(r'(GCF_\d+\.\d+)', filename)
    return match.group(1) if match else None


def sanitize_species_name(species_name):
    """ species_name forfilenameformat( )"""
    sanitized = re.sub(r'[\s/\\:*-"<>|]+', '_', species_name)
    return sanitized


def find_annotation_files_for_species(species_name, gtf_dir, gff_dir):
    """
     species namefound annotation file
    return: {
    'gtf_files': [(file_path, gcf_accession), ...],
    'gff_files': [(file_path, gcf_accession), ...]
    }
    """
    # species name( process )
    species_patterns = [
        species_name,
        sanitize_species_name(species_name),
        species_name.replace(' ', '_'),
        species_name.replace('_', ' ')
    ]
    gtf_files = []
    gff_files = []
    # GTFfile
    for gtf_file in gtf_dir.glob("*.gtf*"):
        filename = gtf_file.name
        # checkfilename species name
        for pattern in species_patterns:
            if filename.startswith(pattern + '_'):
                gcf = extract_gcf_from_filename(filename)
                if gcf:
                    gtf_files.append((gtf_file, gcf))
                    break
                # GFFfile
                for gff_file in gff_dir.glob("*.gff*"):
                    filename = gff_file.name
                    # checkfilename species name
                    for pattern in species_patterns:
                        if filename.startswith(pattern + '_'):
                            gcf = extract_gcf_from_filename(filename)
                            if gcf:
                                gff_files.append((gff_file, gcf))
                                break
                            return {
                            'gtf_files': gtf_files,
                            'gff_files': gff_files
                        }


def main():
    print("=" * 80)
    print("extract WP_protein Assemblyinformation")
    print("=" * 80)
    # 1. readmissing_wp_proteins.tsv
    print("\n[1/5] read WP_proteinlist...")
    if not MISSING_WP_FILE.exists():
        print(f"error: file does not exist: {MISSING_WP_FILE}")
        return
    missing_wp_df = pd.read_csv(MISSING_WP_FILE, sep='\t')
    print(f" load {len(missing_wp_df)} WP_protein")
    # extractall species name(fromGTFandGFFcolumnin)
    all_species = set()
    for _, row in missing_wp_df.iterrows():
        # GTFspecieslist
        if pd.notna(row.get('gtf_species', '')):
            gtf_species = [s.strip() for s in str(row['gtf_species']).split(',') if s.strip()]
            all_species.update(gtf_species)
            # GFFspecieslist
            if pd.notna(row.get('gff_species', '')):
                gff_species = [s.strip() for s in str(row['gff_species']).split(',') if s.strip()]
                all_species.update(gff_species)
                print(f" speciesSummary: {len(all_species)}")
                # 2. readfna_annotation_mapping.tsv( species annotation file )
                print("\n[2/5] readFNA-annotation file mapping ...")
                if not FNA_MAPPING.exists():
                    print(f"error: mapping file does not exist: {FNA_MAPPING}")
                    return
                fna_mapping_df = pd.read_csv(FNA_MAPPING, sep='\t')
                print(f" load {len(fna_mapping_df)} mapping ")
                # 3. found all annotation file andGCF
                print("\n[3/5] find annotation file...")
                species_annotation_info = []
                all_gcf_accessions = set()
                for species in sorted(all_species):
                    # frommappingfileinfind species record
                    species_records = fna_mapping_df[fna_mapping_df['species_name'] == species]
                    if len(species_records) == 0:
                        print(f" warning: not foundspecies {species} mappingrecord")
                        continue
                    for _, record in species_records.iterrows():
                        annotation_file = record['original_annotation_file']
                        annotation_type = record['annotation_type']
                        fna_file = record['fna_file']
                        # extractGCF
                        gcf = extract_gcf_from_filename(annotation_file)
                        if gcf:
                            all_gcf_accessions.add(gcf)
                            species_annotation_info.append({
                                'species_name': species,
                                'fna_file': fna_file,
                                'annotation_type': annotation_type,
                                'annotation_file': annotation_file,
                                'gcf_accession': gcf
                            })
                            print(f" found {len(species_annotation_info)} annotationfilerecord")
                            print(f" {len(all_gcf_accessions)} uniqueGCF ")
                            # save species-annotation file
                            if species_annotation_info:
                                species_annotation_df = pd.DataFrame(species_annotation_info)
                                species_annotation_df = species_annotation_df.sort_values(['species_name', 'annotation_type'])
                                species_annotation_df.to_csv(OUTPUT_SPECIES_ANNOTATION, sep='\t', index=False)
                                print(f" OK save species-annotation fileSummary: {OUTPUT_SPECIES_ANNOTATION}")
                                # 4. readassemblyinformation extract
                                print("\n[4/5] readAssemblyinformation extract ...")
                                if not ASSEMBLY_INFO.exists():
                                    print(f"error: Assemblyinformationfile does not exist: {ASSEMBLY_INFO}")
                                    return
                                assembly_df = pd.read_excel(ASSEMBLY_INFO)
                                print(f" AssemblyrecordSummary: {len(assembly_df)}")
                                # extract - matchedGCF
                                assembly_subset = assembly_df[
                                    assembly_df['assembly_accession'].isin(all_gcf_accessions)
                                ].copy()
                                print(f" matchedtoAssemblyrecordSummary: {len(assembly_subset)}")
                                if len(assembly_subset) == 0:
                                    print(" warning: withmatchedto Assemblyrecord!")
                                    print(" checkassembly_accessioncolumn names correct...")
                                    print(f" availablecolumn: {assembly_df.columns.tolist()}")
                                else:
                                    # save
                                    assembly_subset.to_excel(OUTPUT_ASSEMBLY_SUBSET, index=False)
                                    print(f" OK saveAssemblySummary: {OUTPUT_ASSEMBLY_SUBSET}")
                                    # 5. generatereport
                                    print("\n[5/5] generatereport...")
                                    # statistics species WPcount
                                    species_missing_count = defaultdict(lambda: {'gtf': 0, 'gff': 0, 'wp_list': set()})
                                    for _, row in missing_wp_df.iterrows():
                                        wp_id = row['wp_protein_id']
                                        # GTFspecies
                                        if pd.notna(row.get('gtf_species', '')):
                                            gtf_species = [s.strip() for s in str(row['gtf_species']).split(',') if s.strip()]
                                            for species in gtf_species:
                                                species_missing_count[species]['gtf'] += 1
                                                species_missing_count[species]['wp_list'].add(wp_id)
                                                # GFFspecies
                                                if pd.notna(row.get('gff_species', '')):
                                                    gff_species = [s.strip() for s in str(row['gff_species']).split(',') if s.strip()]
                                                    for species in gff_species:
                                                        species_missing_count[species]['gff'] += 1
                                                        species_missing_count[species]['wp_list'].add(wp_id)
                                                        # create record
                                                        summary_records = []
                                                        for species in sorted(species_missing_count.keys()):
                                                            info = species_missing_count[species]
                                                            # fromspecies_annotation_infoin GCF
                                                            species_gcfs = [item['gcf_accession'] for item in species_annotation_info if item['species_name'] == species]
                                                            gcf_list = ','.join(sorted(set(species_gcfs)))
                                                            summary_records.append({
                                                                'species_name': species,
                                                                'missing_wp_in_gtf': info['gtf'],
                                                                'missing_wp_in_gff': info['gff'],
                                                                'total_unique_missing_wp': len(info['wp_list']),
                                                                'gcf_accessions': gcf_list,
                                                                'missing_wp_list': ','.join(sorted(info['wp_list']))
                                                            })
                                                            summary_df = pd.DataFrame(summary_records)
                                                            summary_df = summary_df.sort_values('total_unique_missing_wp', ascending=False)
                                                            summary_df.to_csv(OUTPUT_SUMMARY, sep='\t', index=False)
                                                            print(f" OK save report: {OUTPUT_SUMMARY}")
                                                            # 6. statistics
                                                            print(f"\n{'=' * 80}")
                                                            print("Processing completed!")
                                                            print(f"{'=' * 80}")
                                                            print(f"\nStatistics: statistics:")
                                                            print(f" - WP_proteinSummary: {len(missing_wp_df)}")
                                                            print(f" - speciesSummary: {len(all_species)}")
                                                            print(f" - GCFSummary: {len(all_gcf_accessions)}")
                                                            print(f" - matchedtoAssemblyrecordSummary: {len(assembly_subset)}")
                                                            if len(summary_df) > 0:
                                                                print(f"\nStatistics: WP first5species:")
                                                                top5 = summary_df.head(5)
                                                                for _, row in top5.iterrows():
                                                                    print(f" - {row['species_name']}: {row['total_unique_missing_wp']} WP")
                                                                    # check withGCF matchedtoAssemblyinformation
                                                                    unmatched_gcfs = all_gcf_accessions - set(assembly_subset['assembly_accession'])
                                                                    if unmatched_gcfs:
                                                                        print(f"\nWarning warning: {len(unmatched_gcfs)} GCF Assemblyinformationinfound:")
                                                                        for gcf in sorted(list(unmatched_gcfs))[:5]:
                                                                            print(f" - {gcf}")
                                                                            if len(unmatched_gcfs) > 5:
                                                                                print(f" ... more {len(unmatched_gcfs) - 5} ")
                                                                                print(f"\nFiles: Output files:")
                                                                                print(f" - AssemblySummary: {OUTPUT_ASSEMBLY_SUBSET}")
                                                                                print(f" - species-annotation fileSummary: {OUTPUT_SPECIES_ANNOTATION}")
                                                                                print(f" - report: {OUTPUT_SUMMARY}")
                                                                                print(f"\nNote: Summary:")
                                                                                print(f" - {OUTPUT_ASSEMBLY_SUBSET} all WPspecies Assemblyinformation")
                                                                                print(f" - {OUTPUT_SPECIES_ANNOTATION} species annotation file andGCF ")
                                                                                print(f" - {OUTPUT_SUMMARY} WPcount, species")


if __name__ == "__main__":
    main()

# New FASTA files

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Extract missing WP_ protein sequences from downloaded protein sequence files
Efficiently process many compressed files with parallel support
"""

import gzip
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd
from multiprocessing import Pool, Manager
from tqdm import tqdm

# Path configuration
BASE_DIR = Path("/path/to/project/data/genome_annotation")
PROTEIN_SEQ_DIR = BASE_DIR / "protein_seq/faa"
MISSING_WP_FILE = BASE_DIR / "report2" / "missing_wp_proteins.tsv"

# Output files
OUTPUT_DIR = BASE_DIR / "protein_seq"
OUTPUT_FASTA = OUTPUT_DIR / "missing_wp_proteins.fa"
OUTPUT_FOUND_LIST = OUTPUT_DIR / "found_wp_proteins.tsv"
OUTPUT_NOT_FOUND_LIST = OUTPUT_DIR / "not_found_wp_proteins.tsv"
OUTPUT_STATS = OUTPUT_DIR / "extraction_stats.tsv"

# Number of parallel cores
NUM_CORES = 64


def extract_wp_id_from_header(header):
    """
    Extract the WP_ protein ID from the FASTA header
    Support multiple formats:
    >WP_000001234.1 description
    >sp|WP_000001234.1|description
    >gi|xxx|ref|WP_000001234.1|description
    """
    # Match WP_-format protein IDs
    match = re.search(r'(WP_\d+\.\d+)', header)
    return match.group(1) if match else None


def extract_gcf_from_filename(filename):
    """Extract the GCF accession from the filename"""
    match = re.search(r'(GCF_\d+\.\d+)', filename)
    return match.group(1) if match else None


def read_fasta_sequences(fasta_path, target_wp_set):
    """
    Read target WP_ protein sequences from one FASTA file
    return: {wp_id: (header, sequence), ...}
    """
    found_sequences = {}
    try:
        # Check whether the file is compressed
        if str(fasta_path).endswith('.gz'):
            open_func = gzip.open
            mode = 'rt'
        else:
            open_func = open
            mode = 'r'
            with open_func(fasta_path, mode) as f:
                current_header = None
                current_seq = []
                current_wp_id = None
                for line in f:
                    line = line.strip()
                    if line.startswith('>'):
                        # Save the previous sequence
                        if current_wp_id and current_wp_id in target_wp_set:
                            sequence = ''.join(current_seq)
                            found_sequences[current_wp_id] = (current_header, sequence)
                            # Start a new sequence
                            current_header = line
                            current_seq = []
                            current_wp_id = extract_wp_id_from_header(line)
                        else:
                            # Accumulate sequence
                            current_seq.append(line)
                            # save after sequence
                            if current_wp_id and current_wp_id in target_wp_set:
                                sequence = ''.join(current_seq)
                                found_sequences[current_wp_id] = (current_header, sequence)
    except Exception as e:
        print(f"Error reading {fasta_path}: {e}")
        return found_sequences


def process_protein_file(args):
    """
    process proteinsfile
    args: (fasta_path, target_wp_set)
    """
    fasta_path, target_wp_set = args
    gcf = extract_gcf_from_filename(fasta_path.name)
    found_sequences = read_fasta_sequences(fasta_path, target_wp_set)
    return {
        'gcf': gcf,
        'file': fasta_path.name,
        'found_count': len(found_sequences),
        'sequences': found_sequences
    }


def main():
    print("=" * 80)
    print("fromproteinsequencefileinextract WP_protein")
    print("=" * 80)
    # 1. read WP_proteinlist
    print("\n[1/5] read WP_proteinlist...")
    if not MISSING_WP_FILE.exists():
        print(f"error: file does not exist: {MISSING_WP_FILE}")
        return
    missing_wp_df = pd.read_csv(MISSING_WP_FILE, sep='\t')
    print(f" load {len(missing_wp_df)} WP_protein")
    # create targetWP_protein
    target_wp_set = set(missing_wp_df['wp_protein_id'].tolist())
    print(f" targetWP_proteinSummary: {len(target_wp_set)}")
    # 2. proteinsequencefile
    print("\n[2/5] proteinsequencefile...")
    if not PROTEIN_SEQ_DIR.exists():
        print(f"error: proteinsequencedirectory does not exist: {PROTEIN_SEQ_DIR}")
        return
    # findallproteinfile
    protein_files = list(PROTEIN_SEQ_DIR.glob("*_protein.faa*"))
    print(f" found {len(protein_files)} proteinssequencefile")
    if len(protein_files) == 0:
        print(" error: with found proteinsequencefile!")
        return
    # 3. rowsextractsequence
    print(f"\n[3/5] {NUM_CORES} rowsextractsequence...")
    # Processing section
    process_args = [(fasta_file, target_wp_set) for fasta_file in protein_files]
    # rows process
    all_results = []
    with Pool(NUM_CORES) as pool:
        for result in tqdm(pool.imap_unordered(process_protein_file, process_args),
            total=len(process_args), desc=" extract "):
        if result and result['found_count'] > 0:
            all_results.append(result)
            print(f" from {len(all_results)} filesinfound sequence")
            # 4. all found sequence
            print("\n[4/5] sequence save...")
            all_found_sequences = {}
            file_stats = []
            for result in all_results:
                all_found_sequences.update(result['sequences'])
                file_stats.append({
                    'gcf_accession': result['gcf'],
                    'file_name': result['file'],
                    'found_wp_count': result['found_count'],
                    'wp_ids': ','.join(sorted(result['sequences'].keys()))
                })
                print(f" found {len(all_found_sequences)} uniqueWP_proteinsequence")
                # savesequencetoFASTAfile
                with open(OUTPUT_FASTA, 'w') as f:
                    for wp_id in sorted(all_found_sequences.keys()):
                        header, sequence = all_found_sequences[wp_id]
                        f.write(f"{header}\n")
                        # 80 rows( FASTAformat)
                        for i in range(0, len(sequence), 80):
                            f.write(f"{sequence[i:i+80]}\n")
                            print(f" OK savesequenceto: {OUTPUT_FASTA}")
                            # savefoundWPlist
                            found_wp_records = []
                            for wp_id, (header, sequence) in all_found_sequences.items():
                                found_wp_records.append({
                                    'wp_protein_id': wp_id,
                                    'sequence_length': len(sequence),
                                    'header': header
                                })
                                found_wp_df = pd.DataFrame(found_wp_records)
                                found_wp_df = found_wp_df.sort_values('wp_protein_id')
                                found_wp_df.to_csv(OUTPUT_FOUND_LIST, sep='\t', index=False)
                                print(f" OK savefoundWPlist: {OUTPUT_FOUND_LIST}")
                                # not foundWP
                                not_found_wp = target_wp_set - set(all_found_sequences.keys())
                                if not_found_wp:
                                    not_found_records = []
                                    for wp_id in sorted(not_found_wp):
                                        # from datain WPinformation
                                        wp_info = missing_wp_df[missing_wp_df['wp_protein_id'] == wp_id].iloc[0]
                                        not_found_records.append({
                                            'wp_protein_id': wp_id,
                                            'in_gtf': wp_info.get('in_gtf', False),
                                            'in_gff': wp_info.get('in_gff', False),
                                            'gtf_species_count': wp_info.get('gtf_species_count', 0),
                                            'gff_species_count': wp_info.get('gff_species_count', 0)
                                        })
                                        not_found_df = pd.DataFrame(not_found_records)
                                        not_found_df.to_csv(OUTPUT_NOT_FOUND_LIST, sep='\t', index=False)
                                        print(f" Warning not found {len(not_found_wp)} WPsequence: {OUTPUT_NOT_FOUND_LIST}")
                                        # savefilestatistics
                                        if file_stats:
                                            file_stats_df = pd.DataFrame(file_stats)
                                            file_stats_df = file_stats_df.sort_values('found_wp_count', ascending=False)
                                            file_stats_df.to_csv(OUTPUT_STATS, sep='\t', index=False)
                                            print(f" OK savefilestatistics: {OUTPUT_STATS}")
                                            # 5. statistics
                                            print(f"\n{'=' * 80}")
                                            print("extractcompleted!")
                                            print(f"{'=' * 80}")
                                            print(f"\nStatistics: statistics:")
                                            print(f" - targetWP_proteinSummary: {len(target_wp_set):,}")
                                            print(f" - successfoundWPSummary: {len(all_found_sequences):,}")
                                            print(f" - not foundWPSummary: {len(not_found_wp):,}")
                                            print(f" - successSummary: {len(all_found_sequences)/len(target_wp_set)*100:.2f}%")
                                            print(f"\nStatistics: sequenceinformation:")
                                            if found_wp_records:
                                                seq_lengths = [r['sequence_length'] for r in found_wp_records]
                                                print(f" - sequenceSummary: {min(seq_lengths)} aa")
                                                print(f" - sequenceSummary: {max(seq_lengths)} aa")
                                                print(f" - average sequenceSummary: {sum(seq_lengths)/len(seq_lengths):.1f} aa")
                                                print(f" - Summary: {sum(seq_lengths):,} aa")
                                                print(f"\nStatistics: filestatistics:")
                                                print(f" - fileSummary: {len(protein_files)}")
                                                print(f" - targetWPfileSummary: {len(all_results)}")
                                                if file_stats:
                                                    top_files = sorted(file_stats, key=lambda x: x['found_wp_count'], reverse=True)[:5]
                                                    print(f"\n found WPfirst5files:")
                                                    for stat in top_files:
                                                        print(f" - {stat['gcf_accession']}: {stat['found_wp_count']} WP")
                                                        print(f"\nFiles: Output files:")
                                                        print(f" - WPproteinsequence: {OUTPUT_FASTA} ({len(all_found_sequences):,} sequence)")
                                                        print(f" - foundWPlist: {OUTPUT_FOUND_LIST}")
                                                        if not_found_wp:
                                                            print(f" - not foundWPlist: {OUTPUT_NOT_FOUND_LIST}")
                                                            print(f" - filestatistics: {OUTPUT_STATS}")
                                                            if not_found_wp:
                                                                print(f"\nWarning Summary: with {len(not_found_wp):,} WPnot found")
                                                                print(f" Possible reason:")
                                                                print(f" 1. GCFproteinfile ")
                                                                print(f" 2. WP_ID fileinformat ")
                                                                print(f" 3. WP Updateor ")
                                                                print(f"\n not foundWP (first 10):")
                                                                for wp_id in sorted(list(not_found_wp))[:10]:
                                                                    print(f" - {wp_id}")
                                                                    if len(not_found_wp) > 10:
                                                                        print(f" ... more {len(not_found_wp) - 10:,} ")
                                                                    else:
                                                                        print(f"\nOK !found all targetWP_proteinsequence!")
                                                                        print(f"\nNote: Summary:")
                                                                        print(f" - {OUTPUT_FASTA} Update protein ")
                                                                        print(f" - 'cat custom_protein_V3.fa {OUTPUT_FASTA} > custom_protein_V4.fa' ")
                                                                        print(f" - or makeblastdb builddata ")


if __name__ == "__main__":
    main()

# Data partitioning

## PathSeq result partitioning

In [ ]:
import pandas as pd
import os
import shutil
from pathlib import Path
from collections import defaultdict

# Path configuration
csv_path = "/path/to/project/data_V3/meta/meta_2007.csv"
source_dir = "/path/to/project/data/pathseq_V3"
base_dir = "/path/to/project/data_V3/cancers_V1"
special_dir = "/path/to/project/data_V3/special"
control_dir = "/path/to/project/data_V3/control"

# Define control sample mapping
control_samples = {
    'brain': [
    'SRR16760091', 'SRR16760085', 'SRR16760084', 'SRR16760083', 'SRR16760080', 'SRR16760089', 'SRR16760090', 'SRR16760100',
    'SRR16760093', 'SRR16760133', 'SRR16760092', 'SRR16760111',
    'SRR16760087', 'SRR16760082', 'SRR16760081', 'SRR16760086',
    'SRR16760122'
],
    'GBM_Normal': [
    'GBM_31', 'GBM_32', 'GBM_33', 'GBM_34', 'GBM_35',
    'GBM_36', 'GBM_37', 'GBM_38', 'GBM_39'
],
    'testicle_6': [
    'SRR12625570', 'SRR16760113', 'SRR12625569',
    'SRR16760114', 'SRR16760112', 'SRR12625571'
]
}

# Define special sample prefix rules
special_rules = {
    'CRC/R': ['B1', 'B4', 'B7', 'E5'],
    'CRC/NR': ['B2', 'B3', 'B6', 'C6'],
    'CRC_MS': ['FXC-T-RNA', 'YDR-T-RNA', 'WJA-T-RNA']
}

def check_special_sample(filename):
    """check for sample, returntargetpathorNone"""
    for special_path, prefixes in special_rules.items():
        for prefix in prefixes:
            if filename.startswith(prefix):
                return os.path.join(special_dir, special_path)
            return None

def get_user_choice():
    """ coveragedoes not existfile"""
    print("\n" + "="*60)
    print("Overwrite options:")
    print(" 1. Overwrite all existing files")
    print(" 2. Skip all existing files")
    print(" 3. Ask before overwriting each file")
    print("="*60)
    while True:
        choice = input("Choose (1/2/3): ").strip()
        if choice in ['1', '2', '3']:
            return choice
        print("Invalid choice, enter 1, 2, or 3")

def should_copy_file(target_file, overwrite_mode, filename):
    """Decide whether the file should be copied"""
    if not os.path.exists(target_file):
        return True
    if overwrite_mode == '1': # overwrite all
        return True
    elif overwrite_mode == '2': # skip all
        return False
    else: # ask each time
        while True:
            choice = input(f" file does not exist: {filename}, coverage- (y/n): ").strip().lower()
            if choice in ['y', 'yes', 'n', 'no']:
                return choice in ['y', 'yes']
            print(" no, input y or n")

def count_files_in_directory(directory, file_extension):
    """statisticsdirectoryin file count"""
    if not os.path.exists(directory):
        return 0
    count = 0
    for root, dirs, files in os.walk(directory):
        count += sum(1 for f in files if f.endswith(file_extension))
        return count

def copy_files():
    """copy filestotargetdirectory"""
    if not os.path.exists(source_dir):
        print(f"error: source directory does not exist {source_dir}")
        return
    # readCSVfile
    print(" readCSVfile...")
    df = pd.read_csv(csv_path)
    print(f" read {len(df)} sample information")
    # create sample mapping
    sample_mapping = {}
    for _, row in df.iterrows():
        sample_name = row['Sample']
        sample_mapping[sample_name] = {
            'cancer_type': row['Cancer_Type'],
            'status': row['Status']
        }
        # createcontrolsample mapping
        control_mapping = {}
        for category, samples in control_samples.items():
            for sample in samples:
                control_mapping[sample] = category
                # all sample
                all_known_samples = set(sample_mapping.keys()) | set(control_mapping.keys())
                # special_rulesinsamplefirst sample
                special_prefixes = []
                for prefixes in special_rules.values():
                    special_prefixes.extend(prefixes)
                    # directory all file
                    source_files = os.listdir(source_dir)
                    bam_files = [f for f in source_files if f.endswith('_rna_output.pathseq.bam')]
                    txt_files = [f for f in source_files if f.endswith('_rna_output.pathseq.txt')]
                    print(f"\nfound {len(bam_files)} BAMfile")
                    print(f"found {len(txt_files)} TXTfile")
                    # extractallsample (deduplicate)
                    all_source_samples = set()
                    for filename in bam_files + txt_files:
                        if filename.endswith('_rna_output.pathseq.bam'):
                            sample_name = filename.replace('_rna_output.pathseq.bam', '')
                        else:
                            sample_name = filename.replace('_rna_output.pathseq.txt', '')
                            all_source_samples.add(sample_name)
                            # unmatched samples
                            unmatched_samples = []
                            for sample in all_source_samples:
                                is_matched = False
                                # check CSVin
                                if sample in sample_mapping:
                                    is_matched = True
                                    # check controlin
                                elif sample in control_mapping:
                                    is_matched = True
                                    # check matchedspecial_rulesfirst
                                else:
                                    for prefix in special_prefixes:
                                        if sample.startswith(prefix):
                                            is_matched = True
                                            break
                                        if not is_matched:
                                            unmatched_samples.append(sample)
                                            # matched sample statistics
                                            print("\n" + "="*60)
                                            print(" matched sample statistics:")
                                            print(f" directory sample count: {len(all_source_samples)}")
                                            print(f" matched sample count: {len(unmatched_samples)}")
                                            if unmatched_samples:
                                                print(f"\n unmatched sampleslist:")
                                                for sample in sorted(unmatched_samples):
                                                    print(f" - {sample}")
                                                    print("="*60)
                                                    # Overwrite options
                                                    overwrite_mode = get_user_choice()
                                                    # statistics
                                                    copied_count = 0
                                                    special_count = 0
                                                    control_count = 0
                                                    skipped_count = 0
                                                    # recordeachtargetdirectory statistics
                                                    transfer_stats = defaultdict(lambda: {'expected': 0, 'copied': 0, 'skipped': 0})
                                                    # processallfile
                                                    all_files = bam_files + txt_files
                                                    print("\nstartcopy files...")
                                                    print("="*60)
                                                    for filename in all_files:
                                                        # extractsample
                                                        if filename.endswith('_rna_output.pathseq.bam'):
                                                            sample_name = filename.replace('_rna_output.pathseq.bam', '')
                                                            file_type = 'bam'
                                                            subfolder = 'bam_V3'
                                                        else:
                                                            sample_name = filename.replace('_rna_output.pathseq.txt', '')
                                                            file_type = 'txt'
                                                            subfolder = 'abundance_pathseq_V3'
                                                            source_file = os.path.join(source_dir, filename)
                                                            # check for sample
                                                            special_path = check_special_sample(sample_name)
                                                            if special_path:
                                                                # sample process
                                                                target_dir = os.path.join(special_path, subfolder)
                                                                os.makedirs(target_dir, exist_ok=True)
                                                                target_file = os.path.join(target_dir, filename)
                                                                stat_key = f"special/{os.path.basename(special_path)}/{subfolder}"
                                                                transfer_stats[stat_key]['expected'] += 1
                                                                if should_copy_file(target_file, overwrite_mode, filename):
                                                                    try:
                                                                        shutil.copy2(source_file, target_file)
                                                                        print(f"OK [ ] {filename} -> {stat_key}")
                                                                        special_count += 1
                                                                        transfer_stats[stat_key]['copied'] += 1
                                                                    except Exception as e:
                                                                        print(f"Failed copyfailed {filename}: {e}")
                                                                    else:
                                                                        print(f"- [skip] {filename}")
                                                                        skipped_count += 1
                                                                        transfer_stats[stat_key]['skipped'] += 1
                                                                elif sample_name in control_mapping:
                                                                    # Controlsampleprocess
                                                                    category = control_mapping[sample_name]
                                                                    target_dir = os.path.join(control_dir, category, subfolder)
                                                                    os.makedirs(target_dir, exist_ok=True)
                                                                    target_file = os.path.join(target_dir, filename)
                                                                    stat_key = f"control/{category}/{subfolder}"
                                                                    transfer_stats[stat_key]['expected'] += 1
                                                                    if should_copy_file(target_file, overwrite_mode, filename):
                                                                        try:
                                                                            shutil.copy2(source_file, target_file)
                                                                            print(f"OK [Control] {filename} -> {stat_key}")
                                                                            control_count += 1
                                                                            transfer_stats[stat_key]['copied'] += 1
                                                                        except Exception as e:
                                                                            print(f"Failed copyfailed {filename}: {e}")
                                                                        else:
                                                                            print(f"- [skip] {filename}")
                                                                            skipped_count += 1
                                                                            transfer_stats[stat_key]['skipped'] += 1
                                                                    elif sample_name in sample_mapping:
                                                                        # sample process
                                                                        cancer_type = sample_mapping[sample_name]['cancer_type']
                                                                        status = sample_mapping[sample_name]['status']
                                                                        target_dir = os.path.join(base_dir, cancer_type, status, subfolder)
                                                                        os.makedirs(target_dir, exist_ok=True)
                                                                        target_file = os.path.join(target_dir, filename)
                                                                        stat_key = f"{cancer_type}/{status}/{subfolder}"
                                                                        transfer_stats[stat_key]['expected'] += 1
                                                                        if should_copy_file(target_file, overwrite_mode, filename):
                                                                            try:
                                                                                shutil.copy2(source_file, target_file)
                                                                                print(f"OK {filename} -> {stat_key}")
                                                                                copied_count += 1
                                                                                transfer_stats[stat_key]['copied'] += 1
                                                                            except Exception as e:
                                                                                print(f"Failed copyfailed {filename}: {e}")
                                                                            else:
                                                                                print(f"- [skip] {filename}")
                                                                                skipped_count += 1
                                                                                transfer_stats[stat_key]['skipped'] += 1
                                                                                # outputdetailedstatistics
                                                                                print("\n" + "="*60)
                                                                                print(" completedOverall statistics:")
                                                                                print(f" sample copy: {copied_count} files")
                                                                                print(f" sample copy: {special_count} files")
                                                                                print(f" Controlsample copy: {control_count} files")
                                                                                print(f" skipfile: {skipped_count} files")
                                                                                print(f" process: {copied_count + special_count + control_count + skipped_count} files")
                                                                                print("="*60)
                                                                                # eachtargetdirectory
                                                                                print("\ntargetdirectory report:")
                                                                                print("="*60)
                                                                                for stat_key in sorted(transfer_stats.keys()):
                                                                                    stats = transfer_stats[stat_key]
                                                                                    # path
                                                                                    if stat_key.startswith('special/'):
                                                                                        parts = stat_key.split('/')
                                                                                        full_dir = os.path.join(special_dir, parts[1], parts[2])
                                                                                    elif stat_key.startswith('control/'):
                                                                                        parts = stat_key.split('/')
                                                                                        full_dir = os.path.join(control_dir, parts[1], parts[2])
                                                                                    else:
                                                                                        parts = stat_key.split('/')
                                                                                        full_dir = os.path.join(base_dir, parts[0], parts[1], parts[2])
                                                                                        # statisticsactualfile
                                                                                        if 'bam' in stat_key:
                                                                                            actual_count = count_files_in_directory(full_dir, '.bam')
                                                                                        else:
                                                                                            actual_count = count_files_in_directory(full_dir, '.txt')
                                                                                            # Processing section
                                                                                            expected = stats['expected']
                                                                                            copied = stats['copied']
                                                                                            skipped = stats['skipped']
                                                                                            status = "OK matched" if actual_count >= expected else "Warning matched"
                                                                                            print(f"\ndirectory: {stat_key}")
                                                                                            print(f" Summary: {expected} files")
                                                                                            print(f" copy: {copied} files")
                                                                                            print(f" skip: {skipped} files")
                                                                                            print(f" directoryactual: {actual_count} files")
                                                                                            print(f" Summary: {status}")
                                                                                            if actual_count != expected:
                                                                                                print(f" Warning Summary: directoryactualfile ({actual_count}) ({expected}) !")
                                                                                                if actual_count > expected:
                                                                                                    print(f" does not exist {actual_count - expected} file")
                                                                                                    print("\n" + "="*60)
                                                                                                    print("all completed!")
                                                                                                    print("="*60)

if __name__ == "__main__":
    print("startfilecopy ...")
    print("="*60)
    copy_files()

# New sample statistics

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# file path
file1_path = "/path/to/project/results/summary_data/cancer_microbial_data.csv"
file2_path = "/path/to/project/data_V3/meta/meta_V5_2503.xlsx"
output_dir = Path("/path/to/project/data_V3/meta")

# Define cancer-type mapping (only the 11 cancer types present in the first file)
cancer_mapping = {
    'Cervical Cancer': 'CESC',
    'Gastric Cancer': 'STAD',
    'Breast Cancer': 'BRCA',
    'Lung Cancer': 'LUNG',
    'Bladder Cancer': 'BLCA',
    'Liver Cancer': 'LIHC',
    'Oral Squamous Cell Carcinoma': 'OSCC',
    'Colorectal Cancer': 'CRC',
    'Pancreatic Cancer': 'PAAD',
    'Thyroid Cancer': 'THCA',
    'Kidney Cancer': 'KIDNEY'
}

print("=" * 80)
print("Read data files...")
print("=" * 80)

# readfirst file
df1 = pd.read_csv(file1_path)
print(f"\nfirst file (cancer_microbial_data.csv):")
print(f" - total rows: {len(df1)}")
print(f" - column names: {df1.columns.tolist()}")

# readsecond file
df2 = pd.read_excel(file2_path)
print(f"\nsecond file (meta_V5_2503.xlsx):")
print(f" - total rows: {len(df2)}")
print(f" - column names: {df2.columns.tolist()}")

# Standardize the first table
df1_clean = df1[['Cancer_type', 'Status', 'Sample']].copy()
df1_clean.columns = ['Cancer_Type', 'Status', 'Sample']

# Standardize the second table
df2_clean = df2[['Cancer_Type', 'Sample_status', 'Run']].copy()
df2_clean.columns = ['Cancer_Type', 'Status', 'Sample']
# Normalize Status to title case
df2_clean['Status'] = df2_clean['Status'].str.capitalize()

# Keep only rows in the second table that map to the 11 cancer types
df2_clean['Cancer_Type_Code'] = df2_clean['Cancer_Type'].map(cancer_mapping)
df2_filtered = df2_clean[df2_clean['Cancer_Type_Code'].notna()].copy()
df2_filtered['Cancer_Type'] = df2_filtered['Cancer_Type_Code']
df2_filtered = df2_filtered[['Cancer_Type', 'Status', 'Sample']]

print("\n" + "=" * 80)
print("data afterstatistics:")
print("=" * 80)
print(f"\nfirst file after:")
print(f" - sample count: {len(df1_clean)}")
print(f" - cancer typeSummary: {df1_clean['Cancer_Type'].nunique()}")
print(f" - cancer-type list: {sorted(df1_clean['Cancer_Type'].unique())}")

print(f"\nsecond file after(only 11cancer type):")
print(f" - sample count: {len(df2_filtered)}")
print(f" - cancer typeSummary: {df2_filtered['Cancer_Type'].nunique()}")
print(f" - cancer-type list: {sorted(df2_filtered['Cancer_Type'].unique())}")

# second filein filter cancer type
df2_excluded = df2_clean[df2_clean['Cancer_Type_Code'].isna()]
if len(df2_excluded) > 0:
    print(f"\nsecond filein 11cancer type data:")
    print(f" - sample count: {len(df2_excluded)}")
    excluded_cancers = df2_excluded['Cancer_Type'].unique()
    print(f" - cancer type: {list(excluded_cancers)}")

# Create unique identifiers
df1_clean['unique_id'] = df1_clean['Cancer_Type'] + '_' + df1_clean['Status'] + '_' + df1_clean['Sample']
df2_filtered['unique_id'] = df2_filtered['Cancer_Type'] + '_' + df2_filtered['Status'] + '_' + df2_filtered['Sample']

# Find samples in each category
set1 = set(df1_clean['unique_id'])
set2 = set(df2_filtered['unique_id'])

both = set1 & set2 # present in both files
only_in_1 = set1 - set2 # only first file
only_in_2 = set2 - set1 # only second file

print("\n" + "=" * 80)
print("Sample comparison results:")
print("=" * 80)
print(f"\nOverall statistics:")
print(f" - first filesample count: {len(set1)}")
print(f" - second filesample count(11cancer type): {len(set2)}")
print(f" - present in both filessample count: {len(both)}")
print(f" - only first file sample count: {len(only_in_1)}")
print(f" - only second file sample count: {len(only_in_2)}")

# Summarize by cancer type
print("\n" + "=" * 80)
print("Detailed statistics by cancer type:")
print("=" * 80)

all_cancers = sorted(set(df1_clean['Cancer_Type'].unique()) | set(df2_filtered['Cancer_Type'].unique()))

stats_list = []
for cancer in all_cancers:
    df1_cancer = df1_clean[df1_clean['Cancer_Type'] == cancer]
    df2_cancer = df2_filtered[df2_filtered['Cancer_Type'] == cancer]
    set1_cancer = set(df1_cancer['unique_id'])
    set2_cancer = set(df2_cancer['unique_id'])
    both_cancer = set1_cancer & set2_cancer
    only1_cancer = set1_cancer - set2_cancer
    only2_cancer = set2_cancer - set1_cancer
    stats_list.append({
        'Cancer_Type': cancer,
        'File1_Count': len(set1_cancer),
        'File2_Count': len(set2_cancer),
        'Both_Count': len(both_cancer),
        'Only_File1': len(only1_cancer),
        'Only_File2': len(only2_cancer)
    })
    print(f"\n{cancer}:")
    print(f" file1sample count: {len(set1_cancer)}")
    print(f" file2sample count: {len(set2_cancer)}")
    print(f" sample count: {len(both_cancer)}")
    print(f" file1: {len(only1_cancer)}")
    print(f" file2: {len(only2_cancer)}")
    # Processing section
    if len(df1_cancer) > 0:
        print(f" file1Summary: ", dict(df1_cancer['Status'].value_counts()))
        if len(df2_cancer) > 0:
            print(f" file2Summary: ", dict(df2_cancer['Status'].value_counts()))

# Save statistics table
stats_df = pd.DataFrame(stats_list)
stats_file = output_dir / "sample_comparison_stats.csv"
stats_df.to_csv(stats_file, index=False)
print(f"\nstatistics table saved to: {stats_file}")

# Merge and deduplicate
print("\n" + "=" * 80)
print("Generate merged table:")
print("=" * 80)

# Processing section
merged_df = pd.concat([df1_clean, df2_filtered], ignore_index=True)

# deduplicate( record)
merged_df_unique = merged_df.drop_duplicates(subset=['Cancer_Type', 'Status', 'Sample'], keep='first')
merged_df_unique = merged_df_unique[['Cancer_Type', 'Status', 'Sample']].sort_values(['Cancer_Type', 'Status', 'Sample'])

print(f" - total rows before merging: {len(merged_df)}")
print(f" - total rows after deduplication: {len(merged_df_unique)}")
print(f" - cancer typeSummary: {merged_df_unique['Cancer_Type'].nunique()}")

# Show sample counts for each cancer type
print("\neachcancer typesample count:")
cancer_counts = merged_df_unique.groupby('Cancer_Type').size().sort_values(ascending=False)
for cancer, count in cancer_counts.items():
    print(f" {cancer}: {count}")

# Save merged table
merged_file = output_dir / "merged_meta_table.csv"
merged_df_unique.to_csv(merged_file, index=False)
print(f"\nOK merged table saved to: {merged_file}")

# saveonly first fileinsample
if len(only_in_1) > 0:
    df_only1 = df1_clean[df1_clean['unique_id'].isin(only_in_1)][['Cancer_Type', 'Status', 'Sample']]
    only1_file = output_dir / "samples_only_in_file1.csv"
    df_only1.to_csv(only1_file, index=False)
    print(f"OK samples only in file 1 saved to: {only1_file}")

# saveonly second fileinsample
if len(only_in_2) > 0:
    df_only2 = df2_filtered[df2_filtered['unique_id'].isin(only_in_2)][['Cancer_Type', 'Status', 'Sample']]
    only2_file = output_dir / "samples_only_in_file2.csv"
    df_only2.to_csv(only2_file, index=False)
    print(f"OK samples only in file 2 saved to: {only2_file}")

# savepresent in both filessample
if len(both) > 0:
    df_both = df1_clean[df1_clean['unique_id'].isin(both)][['Cancer_Type', 'Status', 'Sample']]
    both_file = output_dir / "samples_in_both_files.csv"
    df_both.to_csv(both_file, index=False)
    print(f"OK present in both filessamplesavedto: {both_file}")

print("\n" + "=" * 80)
print("analyzecompleted!")
print("=" * 80)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# file path
file1_path = "/path/to/project/results/summary_data/cancer_microbial_data.csv"
file2_path = "/path/to/project/data_V3/meta/meta_V5_2503.xlsx"
pathseq_dir = Path("/path/to/project/data/pathseq_V3")
output_dir = Path("/path/to/project/data_V3/meta")

# Define cancer-type mapping (only the 11 cancer types present in the first file)
cancer_mapping = {
    'Cervical Cancer': 'CESC',
    'Gastric Cancer': 'STAD',
    'Breast Cancer': 'BRCA',
    'Lung Cancer': 'LUNG',
    'Bladder Cancer': 'BLCA',
    'Liver Cancer': 'LIHC',
    'Oral Squamous Cell Carcinoma': 'OSCC',
    'Colorectal Cancer': 'CRC',
    'Pancreatic Cancer': 'PAAD',
    'Thyroid Cancer': 'THCA',
    'Kidney Cancer': 'KIDNEY'
}

print("=" * 80)
print(" 1: Read actual sample files under the PathSeq directory")
print("=" * 80)

# Get all sample names that actually exist
actual_samples = set()
if pathseq_dir.exists():
    for file in pathseq_dir.glob("*_rna_output.pathseq.bam"):
        # Extract sample names (remove suffix)
        sample_name = file.name.replace("_rna_output.pathseq.bam", "")
        actual_samples.add(sample_name)
        print(f"OK found {len(actual_samples)} actualsamplefile")
else:
    print(f"Warning warning: PathSeq directory does not exist: {pathseq_dir}")

print("\n" + "=" * 80)
print(" 2: Read data files")
print("=" * 80)

# readfirst file
df1 = pd.read_csv(file1_path)
print(f"\nfirst file (cancer_microbial_data.csv):")
print(f" - total rows: {len(df1)}")
print(f" - column names: {df1.columns.tolist()}")

# readsecond file
df2 = pd.read_excel(file2_path)
print(f"\nsecond file (meta_V5_2503.xlsx):")
print(f" - total rows: {len(df2)}")
print(f" - column names: {df2.columns.tolist()}")

print("\n" + "=" * 80)
print(" 3: and second fileinsample ")
print("=" * 80)

# second file withExperimentcolumn
if 'Experiment' not in df2.columns:
    print("Warning warning: second filein withExperimentcolumn, no rows ")
    df2['Experiment'] = None

# Summarize validation results
run_matched = 0
experiment_matched = 0
no_match = 0
replaced_samples = []

# Create a new column for the final sample name
df2['Sample_Validated'] = df2['Run'].copy()

for idx, row in df2.iterrows():
    run_value = str(row['Run']) if pd.notna(row['Run']) else None
    exp_value = str(row['Experiment']) if pd.notna(row['Experiment']) else None
    # First check whether the Run column appears in actual samples
    if run_value and run_value in actual_samples:
        run_matched += 1
        # If Run does not match, try the Experiment column
    elif exp_value and exp_value in actual_samples:
        df2.at[idx, 'Sample_Validated'] = exp_value
        experiment_matched += 1
        replaced_samples.append({
            'Index': idx,
            'Original_Run': run_value,
            'Replaced_With_Experiment': exp_value,
            'Cancer_Type': row['Cancer_Type']
        })
    else:
        no_match += 1

print(f"\nValidation results:")
print(f" - Run column matched directly: {run_matched}")
print(f" - Run did not match but Experiment matched (replaced): {experiment_matched}")
print(f" - unable to match: {no_match}")

if replaced_samples:
    print(f"\nreplacement details ( {len(replaced_samples)}):")
    for i, item in enumerate(replaced_samples[:10], 1): # show only the first 10
        print(f" {i}. [{item['Cancer_Type']}] {item['Original_Run']} -> {item['Replaced_With_Experiment']}")
        if len(replaced_samples) > 10:
            print(f" ... more {len(replaced_samples) - 10} ")

# save record
if replaced_samples:
    replaced_df = pd.DataFrame(replaced_samples)
    replaced_file = output_dir / "sample_name_replacements.csv"
    replaced_df.to_csv(replaced_file, index=False)
    print(f"\nOK replacement record saved to: {replaced_file}")

# checkunmatched samples
unmatched_samples = df2[~df2['Sample_Validated'].isin(actual_samples)].copy()
if len(unmatched_samples) > 0:
    print(f"\nWarning with {len(unmatched_samples)} samplescorresponding files not found in the PathSeq directory")
    unmatched_file = output_dir / "unmatched_samples.csv"
    unmatched_samples[['Cancer_Type', 'Sample_status', 'Run', 'Experiment', 'Sample_Validated']].to_csv(
        unmatched_file, index=False)
    print(f"OK unmatched sample list saved to: {unmatched_file}")

print("\n" + "=" * 80)
print(" 4: data")
print("=" * 80)

# Standardize the first table
df1_clean = df1[['Cancer_type', 'Status', 'Sample']].copy()
df1_clean.columns = ['Cancer_Type', 'Status', 'Sample']

# Standardize the second table( afterSample_Validatedcolumn)
df2_clean = df2[['Cancer_Type', 'Sample_status', 'Sample_Validated']].copy()
df2_clean.columns = ['Cancer_Type', 'Status', 'Sample']
# Normalize Status to title case
df2_clean['Status'] = df2_clean['Status'].str.capitalize()

# Keep only rows in the second table that map to the 11 cancer types
df2_clean['Cancer_Type_Code'] = df2_clean['Cancer_Type'].map(cancer_mapping)
df2_filtered = df2_clean[df2_clean['Cancer_Type_Code'].notna()].copy()
df2_filtered['Cancer_Type'] = df2_filtered['Cancer_Type_Code']
df2_filtered = df2_filtered[['Cancer_Type', 'Status', 'Sample']]

print(f"\nfirst file after:")
print(f" - sample count: {len(df1_clean)}")
print(f" - cancer typeSummary: {df1_clean['Cancer_Type'].nunique()}")

print(f"\nsecond file after(only 11cancer type):")
print(f" - sample count: {len(df2_filtered)}")
print(f" - cancer typeSummary: {df2_filtered['Cancer_Type'].nunique()}")

# Create unique identifiers
df1_clean['unique_id'] = df1_clean['Cancer_Type'] + '_' + df1_clean['Status'] + '_' + df1_clean['Sample']
df2_filtered['unique_id'] = df2_filtered['Cancer_Type'] + '_' + df2_filtered['Status'] + '_' + df2_filtered['Sample']

# Find samples in each category
set1 = set(df1_clean['unique_id'])
set2 = set(df2_filtered['unique_id'])

both = set1 & set2 # present in both files
only_in_1 = set1 - set2 # only first file
only_in_2 = set2 - set1 # only second file

print("\n" + "=" * 80)
print(" 5: Sample comparison results")
print("=" * 80)
print(f"\nOverall statistics:")
print(f" - first filesample count: {len(set1)}")
print(f" - second filesample count(11cancer type): {len(set2)}")
print(f" - present in both filessample count: {len(both)}")
print(f" - only first file sample count: {len(only_in_1)}")
print(f" - only second file sample count: {len(only_in_2)}")

# Summarize by cancer type
print("\n" + "=" * 80)
print(" 6: Detailed statistics by cancer type")
print("=" * 80)

all_cancers = sorted(set(df1_clean['Cancer_Type'].unique()) | set(df2_filtered['Cancer_Type'].unique()))

stats_list = []
for cancer in all_cancers:
    df1_cancer = df1_clean[df1_clean['Cancer_Type'] == cancer]
    df2_cancer = df2_filtered[df2_filtered['Cancer_Type'] == cancer]
    set1_cancer = set(df1_cancer['unique_id'])
    set2_cancer = set(df2_cancer['unique_id'])
    both_cancer = set1_cancer & set2_cancer
    only1_cancer = set1_cancer - set2_cancer
    only2_cancer = set2_cancer - set1_cancer
    stats_list.append({
        'Cancer_Type': cancer,
        'File1_Count': len(set1_cancer),
        'File2_Count': len(set2_cancer),
        'Both_Count': len(both_cancer),
        'Only_File1': len(only1_cancer),
        'Only_File2': len(only2_cancer)
    })
    print(f"\n{cancer}:")
    print(f" file1sample count: {len(set1_cancer)}")
    print(f" file2sample count: {len(set2_cancer)}")
    print(f" sample count: {len(both_cancer)}")
    print(f" file1: {len(only1_cancer)}")
    print(f" file2: {len(only2_cancer)}")

# Save statistics table
stats_df = pd.DataFrame(stats_list)
stats_file = output_dir / "sample_comparison_stats.csv"
stats_df.to_csv(stats_file, index=False)
print(f"\nOK statistics table saved to: {stats_file}")

# Merge and deduplicate
print("\n" + "=" * 80)
print(" 7: Generate merged table")
print("=" * 80)

# Processing section
merged_df = pd.concat([df1_clean, df2_filtered], ignore_index=True)

# deduplicate( record)
merged_df_unique = merged_df.drop_duplicates(subset=['Cancer_Type', 'Status', 'Sample'], keep='first')
merged_df_unique = merged_df_unique[['Cancer_Type', 'Status', 'Sample']].sort_values(['Cancer_Type', 'Status', 'Sample'])

print(f" - total rows before merging: {len(merged_df)}")
print(f" - total rows after deduplication: {len(merged_df_unique)}")
print(f" - cancer typeSummary: {merged_df_unique['Cancer_Type'].nunique()}")

# Show sample counts for each cancer type
print("\neachcancer typesample count:")
cancer_counts = merged_df_unique.groupby('Cancer_Type').size().sort_values(ascending=False)
for cancer, count in cancer_counts.items():
    print(f" {cancer}: {count}")

# Save merged table
merged_file = output_dir / "merged_meta_table.csv"
merged_df_unique.to_csv(merged_file, index=False)
print(f"\nOK merged table saved to: {merged_file}")

# saveonly first fileinsample
if len(only_in_1) > 0:
    df_only1 = df1_clean[df1_clean['unique_id'].isin(only_in_1)][['Cancer_Type', 'Status', 'Sample']]
    only1_file = output_dir / "samples_only_in_file1.csv"
    df_only1.to_csv(only1_file, index=False)
    print(f"OK samples only in file 1 saved to: {only1_file}")

# saveonly second fileinsample
if len(only_in_2) > 0:
    df_only2 = df2_filtered[df2_filtered['unique_id'].isin(only_in_2)][['Cancer_Type', 'Status', 'Sample']]
    only2_file = output_dir / "samples_only_in_file2.csv"
    df_only2.to_csv(only2_file, index=False)
    print(f"OK samples only in file 2 saved to: {only2_file}")

# savepresent in both filessample
if len(both) > 0:
    df_both = df1_clean[df1_clean['unique_id'].isin(both)][['Cancer_Type', 'Status', 'Sample']]
    both_file = output_dir / "samples_in_both_files.csv"
    df_both.to_csv(both_file, index=False)
    print(f"OK present in both filessamplesavedto: {both_file}")

print("\n" + "=" * 80)
print("analyzecompleted!")
print("=" * 80)
print(f"\ngenerated files:")
print(f" 1. {merged_file} - merged master table")
print(f" 2. {stats_file} - statistics")
if replaced_samples:
    print(f" 3. {output_dir / 'sample_name_replacements.csv'} - sample-name replacement record")
if len(unmatched_samples) > 0:
    print(f" 4. {output_dir / 'unmatched_samples.csv'} - unmatched samples")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# file path
file1_path = "/path/to/project/results/summary_data/cancer_microbial_data.csv"
file2_path = "/path/to/project/data_V3/meta/meta_V5_2503.xlsx"
pathseq_dir = Path("/path/to/project/data/pathseq_V3")
output_dir = Path("/path/to/project/data_V3/meta")

# Define cancer-type mapping (only the 11 cancer types present in the first file)
cancer_mapping = {
    'Cervical Cancer': 'CESC',
    'Gastric Cancer': 'STAD',
    'Breast Cancer': 'BRCA',
    'Lung Cancer': 'LUNG',
    'Bladder Cancer': 'BLCA',
    'Liver Cancer': 'LIHC',
    'Oral Squamous Cell Carcinoma': 'OSCC',
    'Colorectal Cancer': 'CRC',
    'Pancreatic Cancer': 'PAAD',
    'Thyroid Cancer': 'THCA',
    'Kidney Cancer': 'KIDNEY'
}

print("=" * 80)
print(" 1: Read actual sample files under the PathSeq directory")
print("=" * 80)

# Get all sample names that actually exist
actual_samples = set()
if pathseq_dir.exists():
    for file in pathseq_dir.glob("*_rna_output.pathseq.bam"):
        # Extract sample names (remove suffix)
        sample_name = file.name.replace("_rna_output.pathseq.bam", "")
        actual_samples.add(sample_name)
        print(f"OK found {len(actual_samples)} actualsamplefile")
else:
    print(f"Warning warning: PathSeq directory does not exist: {pathseq_dir}")

print("\n" + "=" * 80)
print(" 2: Read data files")
print("=" * 80)

# readfirst file
df1 = pd.read_csv(file1_path)
print(f"\nfirst file (cancer_microbial_data.csv):")
print(f" - total rows: {len(df1)}")
print(f" - column names: {df1.columns.tolist()}")

# readsecond file
df2 = pd.read_excel(file2_path)
print(f"\nsecond file (meta_V5_2503.xlsx):")
print(f" - total rows: {len(df2)}")
print(f" - column names: {df2.columns.tolist()}")

print("\n" + "=" * 80)
print(" 3: and second fileinsample ")
print("=" * 80)

# second file withExperimentcolumn
if 'Experiment' not in df2.columns:
    print("Warning warning: second filein withExperimentcolumn, no rows ")
    df2['Experiment'] = None

# Summarize validation results
run_matched = 0
experiment_matched = 0
no_match = 0
replaced_samples = []

# Create a new column for the final sample name
df2['Sample_Validated'] = df2['Run'].copy()

for idx, row in df2.iterrows():
    run_value = str(row['Run']) if pd.notna(row['Run']) else None
    exp_value = str(row['Experiment']) if pd.notna(row['Experiment']) else None
    # First check whether the Run column appears in actual samples
    if run_value and run_value in actual_samples:
        run_matched += 1
        # If Run does not match, try the Experiment column
    elif exp_value and exp_value in actual_samples:
        df2.at[idx, 'Sample_Validated'] = exp_value
        experiment_matched += 1
        replaced_samples.append({
            'Index': idx,
            'Original_Run': run_value,
            'Replaced_With_Experiment': exp_value,
            'Cancer_Type': row['Cancer_Type']
        })
    else:
        no_match += 1

print(f"\nValidation results:")
print(f" - Run column matched directly: {run_matched}")
print(f" - Run did not match but Experiment matched (replaced): {experiment_matched}")
print(f" - unable to match: {no_match}")

if replaced_samples:
    print(f"\nreplacement details ( {len(replaced_samples)}):")
    for i, item in enumerate(replaced_samples[:10], 1): # show only the first 10
        print(f" {i}. [{item['Cancer_Type']}] {item['Original_Run']} -> {item['Replaced_With_Experiment']}")
        if len(replaced_samples) > 10:
            print(f" ... more {len(replaced_samples) - 10} ")

# save record
if replaced_samples:
    replaced_df = pd.DataFrame(replaced_samples)
    replaced_file = output_dir / "sample_name_replacements.csv"
    replaced_df.to_csv(replaced_file, index=False)
    print(f"\nOK replacement record saved to: {replaced_file}")

# checkunmatched samples
unmatched_samples = df2[~df2['Sample_Validated'].isin(actual_samples)].copy()
if len(unmatched_samples) > 0:
    print(f"\nWarning with {len(unmatched_samples)} samplescorresponding files not found in the PathSeq directory")
    unmatched_file = output_dir / "unmatched_samples.csv"
    unmatched_samples[['Cancer_Type', 'Sample_status', 'Run', 'Experiment', 'Sample_Validated']].to_csv(
        unmatched_file, index=False)
    print(f"OK unmatched sample list saved to: {unmatched_file}")

print("\n" + "=" * 80)
print(" 4: data")
print("=" * 80)

# Standardize the first table
df1_clean = df1[['Cancer_type', 'Status', 'Sample']].copy()
df1_clean.columns = ['Cancer_Type', 'Status', 'Sample']

# Standardize the second table( afterSample_Validatedcolumn)
df2_clean = df2[['Cancer_Type', 'Sample_status', 'Sample_Validated']].copy()
df2_clean.columns = ['Cancer_Type', 'Status', 'Sample']
# Normalize Status to title case
df2_clean['Status'] = df2_clean['Status'].str.capitalize()

# Keep only rows in the second table that map to the 11 cancer types
df2_clean['Cancer_Type_Code'] = df2_clean['Cancer_Type'].map(cancer_mapping)
df2_filtered = df2_clean[df2_clean['Cancer_Type_Code'].notna()].copy()
df2_filtered['Cancer_Type'] = df2_filtered['Cancer_Type_Code']
df2_filtered = df2_filtered[['Cancer_Type', 'Status', 'Sample']]

print(f"\nfirst file after:")
print(f" - sample count: {len(df1_clean)}")
print(f" - cancer typeSummary: {df1_clean['Cancer_Type'].nunique()}")

print(f"\nsecond file after(only 11cancer type):")
print(f" - sample count: {len(df2_filtered)}")
print(f" - cancer typeSummary: {df2_filtered['Cancer_Type'].nunique()}")

# Create unique identifiers
df1_clean['unique_id'] = df1_clean['Cancer_Type'] + '_' + df1_clean['Status'] + '_' + df1_clean['Sample']
df2_filtered['unique_id'] = df2_filtered['Cancer_Type'] + '_' + df2_filtered['Status'] + '_' + df2_filtered['Sample']

# Find samples in each category
set1 = set(df1_clean['unique_id'])
set2 = set(df2_filtered['unique_id'])

both = set1 & set2 # present in both files
only_in_1 = set1 - set2 # only first file
only_in_2 = set2 - set1 # only second file

print("\n" + "=" * 80)
print(" 5: Sample comparison results")
print("=" * 80)
print(f"\nOverall statistics:")
print(f" - first filesample count: {len(set1)}")
print(f" - second filesample count(11cancer type): {len(set2)}")
print(f" - present in both filessample count: {len(both)}")
print(f" - only first file sample count: {len(only_in_1)}")
print(f" - only second file sample count: {len(only_in_2)}")

# Summarize by cancer type
print("\n" + "=" * 80)
print(" 6: Detailed statistics by cancer type")
print("=" * 80)

all_cancers = sorted(set(df1_clean['Cancer_Type'].unique()) | set(df2_filtered['Cancer_Type'].unique()))

stats_list = []
only_file2_by_cancer = {} # eachcancer type file2insample

for cancer in all_cancers:
    df1_cancer = df1_clean[df1_clean['Cancer_Type'] == cancer]
    df2_cancer = df2_filtered[df2_filtered['Cancer_Type'] == cancer]
    set1_cancer = set(df1_cancer['unique_id'])
    set2_cancer = set(df2_cancer['unique_id'])
    both_cancer = set1_cancer & set2_cancer
    only1_cancer = set1_cancer - set2_cancer
    only2_cancer = set2_cancer - set1_cancer
    stats_list.append({
        'Cancer_Type': cancer,
        'File1_Count': len(set1_cancer),
        'File2_Count': len(set2_cancer),
        'Both_Count': len(both_cancer),
        'Only_File1': len(only1_cancer),
        'Only_File2': len(only2_cancer)
    })
    print(f"\n{cancer}:")
    print(f" file1sample count: {len(set1_cancer)}")
    print(f" file2sample count: {len(set2_cancer)}")
    print(f" sample count: {len(both_cancer)}")
    print(f" file1: {len(only1_cancer)}")
    print(f" file2: {len(only2_cancer)}")
    # with coverage file2insample, save
    if len(only2_cancer) > 0:
        df_only2_cancer = df2_cancer[df2_cancer['unique_id'].isin(only2_cancer)][['Cancer_Type', 'Status', 'Sample']].copy()
        only_file2_by_cancer[cancer] = df_only2_cancer
        # Show sample list
        print(f" -> file2sample columnSummary:")
        for idx, row in df_only2_cancer.iterrows():
            print(f" - {row['Sample']} ({row['Status']})")
            # Save by cancer type
            only2_cancer_file = output_dir / f"samples_only_in_file2_{cancer}.csv"
            df_only2_cancer.to_csv(only2_cancer_file, index=False)
            print(f" OK savedto: {only2_cancer_file.name}")

# Save statistics table
stats_df = pd.DataFrame(stats_list)
stats_file = output_dir / "sample_comparison_stats.csv"
stats_df.to_csv(stats_file, index=False)
print(f"\nOK statistics table saved to: {stats_file}")

# Merge and deduplicate
print("\n" + "=" * 80)
print(" 7: Generate merged table")
print("=" * 80)

# Processing section
merged_df = pd.concat([df1_clean, df2_filtered], ignore_index=True)

# deduplicate( record)
merged_df_unique = merged_df.drop_duplicates(subset=['Cancer_Type', 'Status', 'Sample'], keep='first')
merged_df_unique = merged_df_unique[['Cancer_Type', 'Status', 'Sample']].sort_values(['Cancer_Type', 'Status', 'Sample'])

print(f" - total rows before merging: {len(merged_df)}")
print(f" - total rows after deduplication: {len(merged_df_unique)}")
print(f" - cancer typeSummary: {merged_df_unique['Cancer_Type'].nunique()}")

# Show sample counts for each cancer type
print("\neachcancer typesample count:")
cancer_counts = merged_df_unique.groupby('Cancer_Type').size().sort_values(ascending=False)
for cancer, count in cancer_counts.items():
    print(f" {cancer}: {count}")

# Save merged table
merged_file = output_dir / "merged_meta_table.csv"
merged_df_unique.to_csv(merged_file, index=False)
print(f"\nOK merged table saved to: {merged_file}")

# saveonly first fileinsample
if len(only_in_1) > 0:
    df_only1 = df1_clean[df1_clean['unique_id'].isin(only_in_1)][['Cancer_Type', 'Status', 'Sample']]
    only1_file = output_dir / "samples_only_in_file1.csv"
    df_only1.to_csv(only1_file, index=False)
    print(f"OK samples only in file 1 saved to: {only1_file}")

# saveonly second fileinsample
if len(only_in_2) > 0:
    df_only2 = df2_filtered[df2_filtered['unique_id'].isin(only_in_2)][['Cancer_Type', 'Status', 'Sample']]
    only2_file = output_dir / "samples_only_in_file2.csv"
    df_only2.to_csv(only2_file, index=False)
    print(f"OK samples only in file 2 saved to: {only2_file}")

# savepresent in both filessample
if len(both) > 0:
    df_both = df1_clean[df1_clean['unique_id'].isin(both)][['Cancer_Type', 'Status', 'Sample']]
    both_file = output_dir / "samples_in_both_files.csv"
    df_both.to_csv(both_file, index=False)
    print(f"OK present in both filessamplesavedto: {both_file}")

print("\n" + "=" * 80)
print("analyzecompleted!")
print("=" * 80)
print(f"\ngenerated files:")
file_count = 1
print(f" {file_count}. {merged_file.name} - merged master table")
file_count += 1
print(f" {file_count}. {stats_file.name} - statistics")
file_count += 1

if replaced_samples:
    print(f" {file_count}. sample_name_replacements.csv - sample-name replacement record")
    file_count += 1
if len(unmatched_samples) > 0:
    print(f" {file_count}. unmatched_samples.csv - unmatched samples")
    file_count += 1

if len(only_in_1) > 0:
    print(f" {file_count}. samples_only_in_file1.csv - all samples only in file 1")
    file_count += 1
if len(only_in_2) > 0:
    print(f" {file_count}. samples_only_in_file2.csv - all samples only in file 2")
    file_count += 1

if len(both) > 0:
    print(f" {file_count}. samples_in_both_files.csv - present in both filessample")
    file_count += 1

# Save by cancer type" file2"sample file
if only_file2_by_cancer:
    print(f"\n Save by cancer type' file2'sample:")
    for cancer in sorted(only_file2_by_cancer.keys()):
        count = len(only_file2_by_cancer[cancer])
        print(f" - samples_only_in_file2_{cancer}.csv ({count}samples)")

In [ ]:
import os
import gzip
import shutil
from pathlib import Path

# Define sample groups
sample_classification = {
    'BLCA': {
    'Tumor': ['SRR9623637', 'SRR9623641']
},
    'LIHC': {
    'Normal': ['SRR14747909', 'SRR14747911', 'SRR14747913', 'SRR14747917', 'SRR14747919', 'SRR14747921', 'SRR14747923', 'SRR14747925', 'SRR14747927', 'SRR14747929', 'SRR14747933', 'SRR14747935', 'SRR14747937'],
    'Tumor': ['SRR14747908', 'SRR14747910', 'SRR14747912', 'SRR14747914', 'SRR14747916', 'SRR14747918', 'SRR14747920', 'SRR14747922', 'SRR14747924', 'SRR14747928', 'SRR14747930', 'SRR14747932', 'SRR14747934', 'SRR14747936']
}
}

# Source directory and target base directory
source_dir = '/path/to/project/data/flagstat_29'
base_target_dir = '/path/to/project/data/cancers_V1'

# Create a mapping from sample ID to target path
sample_to_path = {}
for cancer, statuses in sample_classification.items():
    for status, samples in statuses.items():
        target_path = os.path.join(base_target_dir, cancer, status, 'flagstat')
        for sample in samples:
            sample_to_path[sample] = target_path

# Create all target directories
for target_path in set(sample_to_path.values()):
    os.makedirs(target_path, exist_ok=True)
    print(f"Create directory: {target_path}")

# Process all gz files
gz_files = list(Path(source_dir).glob('*.gz'))
print(f"\nfound {len(gz_files)} gzfile")

processed = 0
skipped = 0

for gz_file in gz_files:
    # extractsample ( filenameformatfor SRR***.flagstat.gz or )
    filename = gz_file.stem # remove.gzafter
    sample_id = None
    # fromfilenameinextractsampleID
    for sample in sample_to_path.keys():
        if sample in str(gz_file.name):
            sample_id = sample
            break
        if sample_id and sample_id in sample_to_path:
            target_dir = sample_to_path[sample_id]
            # Decompress file
            output_file = os.path.join(target_dir, filename)
            try:
                with gzip.open(gz_file, 'rb') as f_in:
                    with open(output_file, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                        print(f"OK processed: {gz_file.name} -> {target_dir}")
                        processed += 1
            except Exception as e:
                print(f"Failed error while processing {gz_file.name}: {e}")
            else:
                print(f"Warning skip unrecognized file: {gz_file.name}")
                skipped += 1

print(f"\nProcessing completed!")
print(f"processed successfully: {processed} files")
print(f"skip: {skipped} files")

# BRCA R NR

In [ ]:
#!/usr/bin/env python3
import os
import shutil
from pathlib import Path

def extract_sample_name(filename):
    """Extract the sample name from the filename and remove the _rna_output.pathseq.bam suffix"""
    if filename.endswith('_rna_output.pathseq.bam'):
        return filename.replace('_rna_output.pathseq.bam', '')
    return None

def copy_files_for_group(group_name):
    """Copy files for the R or NR group"""
    print(f"\nprocess {group_name} ...")
    # directory: extractsample
    source_bam_dir = f"/path/to/project/data/special/BRCA/{group_name}/bam"
    # copy source directory
    copy_from_bam = "/path/to/project/data_V3/cancers/BRCA/Tumor/bam_V3"
    copy_from_ratio = "/path/to/project/data_V3/cancers/BRCA/Tumor/reads_ratio_V4"
    copy_from_pathseq = "/path/to/project/data_V3/cancers/BRCA/Tumor/abundance_pathseq_V3"
    # targetdirectory
    target_bam_dir = f"/path/to/project/data_V3/special/BRCA/{group_name}/bam_V3"
    target_ratio_dir = f"/path/to/project/data_V3/special/BRCA/{group_name}/reads_ratio_V4"
    target_pathseq_dir = f"/path/to/project/data_V3/special/BRCA/{group_name}/abundance_pathseq_V3"
    # createtargetdirectory
    os.makedirs(target_bam_dir, exist_ok=True)
    os.makedirs(target_ratio_dir, exist_ok=True)
    os.makedirs(target_pathseq_dir, exist_ok=True)
    # check directory does not exist
    if not os.path.exists(source_bam_dir):
        print(f"error: source directory does not exist - {source_bam_dir}")
        return
    # Get all sample names
    samples = []
    for filename in os.listdir(source_bam_dir):
        sample_name = extract_sample_name(filename)
        if sample_name:
            samples.append(sample_name)
            print(f"found {len(samples)} samples: {samples}")
            # forsamplescopy files
            for sample in samples:
                print(f"\nProcess sample: {sample}")
                # copy bam file
                bam_file = f"{sample}_rna_output.pathseq.bam"
                source_bam = os.path.join(copy_from_bam, bam_file)
                target_bam = os.path.join(target_bam_dir, bam_file)
                if os.path.exists(source_bam):
                    shutil.copy2(source_bam, target_bam)
                    print(f" OK copied bam file: {bam_file}")
                else:
                    print(f" Failed bam file not found: {source_bam}")
                    # copy ratio file
                    ratio_file = f"{sample}_non_host_ratio.txt"
                    source_ratio = os.path.join(copy_from_ratio, ratio_file)
                    target_ratio = os.path.join(target_ratio_dir, ratio_file)
                    if os.path.exists(source_ratio):
                        shutil.copy2(source_ratio, target_ratio)
                        print(f" OK copied ratio file: {ratio_file}")
                    else:
                        print(f" Failed ratio file not found: {source_ratio}")

                        # copypathseqfile
                        pathseq_file = f"{sample}_rna_output.pathseq.txt"
                        source_pathseq = os.path.join(copy_from_pathseq, pathseq_file)
                        target_pathseq = os.path.join(target_pathseq_dir, pathseq_file)
                        if os.path.exists(source_pathseq):
                            shutil.copy2(source_pathseq, target_pathseq)
                            print(f" OK copied ratio file: {pathseq_file}")
                        else:
                            print(f" Failed ratio file not found: {source_pathseq}")

def main():
    print("="*60)
    print("BRCA samplefilecopy ")
    print("="*60)
    # processR andNR
    for group in ['R', 'NR']:
        copy_files_for_group(group)
        print("\n" + "="*60)
        print("Processing completed!")
        print("="*60)

if __name__ == "__main__":
    main()